# CS781 Phase 4. Efficient student with prefix distillation

I initialized the same 55.00M-parameter student as the [gold-label control](phase4_efficient_gold.ipynb). I combined 70% class-weighted cross-entropy with 30% temperature-scaled teacher-to-student KL divergence using temperature 2. The teacher logits come from the [initial prefix checkpoint](phase4_prefix_lorma_teacher.ipynb) at update 275.

I reconstructed this notebook from the archived training sources, settings and results. The table reports the saved experiment, and the runnable cells start with empty outputs.

| Selected epoch / update | Threshold | Dev macro F1 | CodaBench macro F1 | Submission ID |
| --- | ---: | ---: | ---: | ---: |
| 5 / 2500 | 0.40 | 0.6514 | 0.6185 | 1009373 |

## 1. Inputs and experiment settings

1. Start a Kaggle session with two T4 GPUs. The saved experiments used PyTorch 2.11.0 with CUDA 12.8.
2. Mount `phase4_cjpe_fast_cache.zip` or `phase4_cjpe_prepared_data.zip` from [Phase 4 preparation](phase4_cjpe_preparation.ipynb). Set `PREPARED_INPUT` to the ZIP or its extracted folder if discovery is ambiguous.
3. Mount the original Phase 1 checkpoint [`efficient_002_mixed217m_500m`](../phase1/phase1_efficient_500m_continuation.ipynb) with its `best_model.pt`. Set `PHASE1_INPUT` to that checkpoint, folder or backup ZIP if needed.
4. Set `TEACHER_CACHE_INPUT` to `cjpe_prefix_lorma_001_teacher_cache.zip` from the update-275 teacher, or to the extracted `teacher_logits_train.npz`. The loader matches logits to the official training rows by row ID.

The configured run ID is `cjpe_efficient_kd_001`.

In [ ]:
from pathlib import Path
import json
import sys

# ---- Run and inputs ----
ARCHITECTURE = "efficient"
STUDENT_MODE = "kd"
RUN_TAG = "001"
ACTION = "train"                  # train / export_only / prepare_only (data bundle only)
PREPARED_INPUT = None             # Fast-cache ZIP/folder or original prepared-data ZIP/folder
DATA_READ_WORKERS = 16            # CPU threads to overlap initial mounted-cache reads
PHASE1_INPUT = None               # Original checkpoint, folder or Phase 1 backup ZIP
RESUME_INPUT = None               # This training run's backup ZIP/folder; None = local resume
TEACHER_CACHE_INPUT = None  # Update-275 prefix teacher cache ZIP, folder or NPZ
AUTO_RESUME = True                # Local latest progress takes precedence over the same backup

# ---- Main training choices ----
STAGE = 1
EPOCHS = 4 if STAGE in (1, 2) else 10
MINIMUM_EPOCHS = 2
PATIENCE = 1 if STAGE in (1, 2) else 2
EARLY_STOP_MIN_DELTA = 0.002  # AUC units
MAX_RUN_HOURS = 10 if STAGE == 3 else None
MAX_UPDATES_THIS_INVOCATION = {1: 1011, 2: 1009, 3: None}[STAGE]
SEED = 42
BACKBONE_LR = {"decoder": 5e-6, "encoder_decoder": 1e-4,
               "prefix": 1e-4, "efficient": 1e-5}[ARCHITECTURE]
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
BACKBONE_DROPOUT = 0.10
HEAD_DROPOUT = 0.30
LABEL_SMOOTHING = 0.0
CLASS_WEIGHTS = "balanced"              # None / "balanced" / "sqrt"
GRADIENT_CLIP = 1.0
ADAM_BETAS = (0.9, 0.999)
ADAM_EPS = 1e-8
WARMUP_FRACTION = 0.10
MIN_LR_FRACTION = 0.01

# ---- Cases and document geometry ----
TRAIN_BATCH_CASES = 8             # Total case batch, split over the two GPUs
EVAL_BATCH_CASES = 8
ACCUMULATION_STEPS = 8            # Effective cases/update = batch * accumulation, except tail
CHUNK_TOKENS = 512                # Includes BOS/EOS; original context limits still apply
CHUNK_OVERLAP = 64                # Content-token overlap
MAX_CHUNKS = 4
CHUNK_POLICY = "tail"          # uniform / head / tail / head_tail
CHUNK_MICROBATCH = 2              # Chunks evaluated together within one case
GRADIENT_CHECKPOINTING = True
AMP = True                       # FP16 autocast + GradScaler on T4
REQUIRE_TWO_GPUS = True
DEDUPLICATE_IDENTICAL_TRAINING_ROWS = True

# ---- Method and common document head ----
DECODER_UPPER_LAYERS = 2
ADAPTER_BOTTLENECK = 64
ADAPTER_DROPOUT = 0.05
ADAPTER_TRAIN_LAYER_NORM = True
LORMA_RANK = 8
LORMA_ALPHA = 8
LORMA_DROPOUT = 0.05
STUDENT_LAYERS = [0, 7, 15]
PROJECTION_DIM = 256
CONTEXT_HIDDEN = 128
ATTENTION_HIDDEN = 64

# ---- Evaluation, saving and short startup ----
EVAL_EVERY_UPDATES = 125           # 0 = epoch-end/pause evaluation only
TRAIN_EVAL_CASES = 512            # Fixed subset at epoch end; None = full training set; 0 disables
LOG_EVERY_UPDATES = 20
SAVE_EVERY_MINUTES = 10
BACKUP_EVERY_MINUTES = 30
RUN_STARTUP_PROBE = True          # Two discarded zero-LR stress updates, not a learning sweep
AUTO_REDUCE_BATCH_ON_STARTUP = True
AMP_RETRIES = 3
TUNE_THRESHOLD = True
THRESHOLD_GRID = [round(0.20 + i*0.01, 2) for i in range(61)]

# ---- Stable evaluation and automatic effective-batch preservation ----
USE_EMA = True                    # Evaluate/export averaged trainable weights
EMA_DECAY = 0.98                  # About 50 optimizer updates of smoothing
SELECTION_METRIC = "roc_auc"      # Checkpoint selection; F1 remains the reported task metric
TARGET_EFFECTIVE_CASES = 64       # Increase accumulation if startup reduces the batch

# ---- KD / optional later teacher export ----
EXPORT_TEACHER_CACHE = False      # Extra full-train inference; enable after choosing the teacher
TEMPERATURE = 2.0
KD_WEIGHT = 0.3                   # loss = (1-w)*gold_CE + w*T^2*KL(teacher || student)

# ---- Paths; everything below just assembles the settings ----
WORK_ROOT = Path("/kaggle/working/cs781_phase4_training")
OUTPUT_ROOT = Path("/kaggle/working")
SOURCE_DIR = Path("/kaggle/working/cs781_phase4_train_src")
METHOD = {"decoder": "partial", "encoder_decoder": "adapters", "prefix": "lorma",
          "efficient": STUDENT_MODE}[ARCHITECTURE]
RUN_ID = f"cjpe_{ARCHITECTURE}_{METHOD}_{RUN_TAG}"
MODEL_SETTINGS = dict(seed=SEED, head_seed=SEED, gradient_checkpointing=GRADIENT_CHECKPOINTING,
    chunk_microbatch=CHUNK_MICROBATCH, backbone_dropout=BACKBONE_DROPOUT, head_dropout=HEAD_DROPOUT,
    decoder_upper_layers=DECODER_UPPER_LAYERS, adapter_bottleneck=ADAPTER_BOTTLENECK,
    adapter_dropout=ADAPTER_DROPOUT, adapter_train_layer_norm=ADAPTER_TRAIN_LAYER_NORM,
    lorma_rank=LORMA_RANK, lorma_alpha=LORMA_ALPHA, lorma_dropout=LORMA_DROPOUT,
    student_layers=STUDENT_LAYERS, projection_dim=PROJECTION_DIM,
    context_hidden=CONTEXT_HIDDEN, attention_hidden=ATTENTION_HIDDEN)
CONFIG = dict(run_id=RUN_ID, architecture=ARCHITECTURE,
    student_mode=STUDENT_MODE if ARCHITECTURE == "efficient" else "supervised", action=ACTION,
    work_root=str(WORK_ROOT), output_root=str(OUTPUT_ROOT), device="cuda:0",
    prepared_input=PREPARED_INPUT, phase1_input=PHASE1_INPUT, resume_input=RESUME_INPUT,
    data_read_workers=DATA_READ_WORKERS,
    auto_resume=AUTO_RESUME, teacher_cache_input=TEACHER_CACHE_INPUT,
    seed=SEED, epochs=EPOCHS, minimum_epochs=MINIMUM_EPOCHS, patience=PATIENCE,
    early_stop_min_delta=EARLY_STOP_MIN_DELTA, max_run_hours=MAX_RUN_HOURS,
    max_updates_this_invocation=MAX_UPDATES_THIS_INVOCATION, backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY, adam_betas=ADAM_BETAS, adam_eps=ADAM_EPS,
    warmup_fraction=WARMUP_FRACTION, min_lr_fraction=MIN_LR_FRACTION, gradient_clip=GRADIENT_CLIP,
    label_smoothing=LABEL_SMOOTHING, class_weights=CLASS_WEIGHTS,
    train_batch_cases=TRAIN_BATCH_CASES, eval_batch_cases=EVAL_BATCH_CASES,
    accumulation_steps=ACCUMULATION_STEPS, total_tokens=CHUNK_TOKENS, overlap=CHUNK_OVERLAP,
    max_chunks=MAX_CHUNKS, policy=CHUNK_POLICY, amp=AMP, require_two_gpus=REQUIRE_TWO_GPUS,
    deduplicate_identical_training_rows=DEDUPLICATE_IDENTICAL_TRAINING_ROWS,
    eval_every_updates=EVAL_EVERY_UPDATES, train_eval_cases=TRAIN_EVAL_CASES,
    log_every_updates=LOG_EVERY_UPDATES, save_every_minutes=SAVE_EVERY_MINUTES,
    backup_every_minutes=BACKUP_EVERY_MINUTES, run_startup_probe=RUN_STARTUP_PROBE,
    auto_reduce_batch_on_startup=AUTO_REDUCE_BATCH_ON_STARTUP, amp_retries=AMP_RETRIES,
    tune_threshold=TUNE_THRESHOLD, threshold_grid=THRESHOLD_GRID,
    export_teacher_cache=EXPORT_TEACHER_CACHE, temperature=TEMPERATURE, kd_weight=KD_WEIGHT,
    use_ema=USE_EMA, ema_decay=EMA_DECAY, selection_metric=SELECTION_METRIC,
    target_effective_cases=TARGET_EFFECTIVE_CASES, model_settings=MODEL_SETTINGS)
print(json.dumps({k: CONFIG[k] for k in ("run_id", "architecture", "student_mode", "action",
    "epochs", "backbone_lr", "head_lr", "train_batch_cases", "accumulation_steps", "max_run_hours")}, indent=2))

## 2. Runtime and implementation

The setup cell uses the installed GPU runtime and fills in missing support
packages. The source cells write the original Phase 1 modules, prepared
data loader, classifier, trainer and course scorer into working storage.
The loader checks the prepared cache and the original checkpoint against
their recorded identities.

In [ ]:
import importlib.util
import os
import subprocess

packages = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import torch
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Visible GPUs:", [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
print("Run folder:", WORK_ROOT/"runs"/RUN_ID)

In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/phase1_utils.py
"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/a1_tokenizer.py
"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/phase1_decoder.py
"""Phase 1 decoder, adapted from completed Assignment 3 Q1."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int = 16000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    norm_first: bool = True
    attention_impl: str = "sdpa"
    attention_scale: float = None
    embedding_scale: float = 32.0


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0,
                 attention_scale=None, attention_impl="sdpa"):
        super().__init__()
        assert d_model % n_heads == 0
        if attention_impl not in ("manual", "sdpa"):
            raise ValueError("attention_impl must be manual or sdpa")
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head) if attention_scale is None else float(attention_scale)
        self.attention_impl = attention_impl
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if attention_impl == "manual":
            self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
                                 persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if self.attention_impl == "manual":
            scores = (q @ k.transpose(-2, -1)) * self.attention_scale
            scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
            weights = self.attn_dropout(torch.softmax(scores, dim=-1))
            out = weights @ v
        else:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True, scale=self.attention_scale,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
            )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * frequency)
    pe[:, 1::2] = torch.cos(position * frequency)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                      cfg.attention_scale, cfg.attention_impl)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                             persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        # parameters() counts the tied matrix once.
        for parameter in self.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        for module in self.modules():
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, last_positions=None):
        B, T = idx.shape
        if T > self.cfg.max_seq_len:
            raise ValueError("Input exceeds the configured context length")
        x = self.tok_emb(idx) * self.cfg.embedding_scale + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if last_positions is not None:
            x = x[torch.arange(B, device=idx.device), last_positions]
        return self.head(x)


class ModelWithLoss(nn.Module):
    """Compute loss on each GPU before DataParallel gathers the small outputs."""
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm = lm
        self.use_amp = use_amp

    def forward(self, x, y):
        # Enter autocast in each worker thread; do not move the model to half().
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits = self.lm(x)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), y.reshape(-1))
        return loss.unsqueeze(0)


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/phase1_shared.py
"""Three from-scratch Phase 1 Transformers; course blocks with cited extensions.

SDPA Boolean masks use True for ALLOWED connections. All blocks are pre-norm.
Training uses full teacher-forced sequences. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "encoder_decoder"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    encoder_layers: int = 8
    decoder_layers: int = 6
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128


def configuration(architecture):
    if architecture not in ("encoder_decoder", "prefix", "efficient"):
        raise ValueError("Unknown architecture")
    return ModelConfig(architecture=architecture, d_ff=4864 if architecture == "efficient" else 4096)


def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


def valid_keys(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, x, allowed):
        x = x + self.attn(self.ln1(x), allowed=allowed)
        return x + self.ffn(self.ln2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2, self.ln3 = (nn.LayerNorm(cfg.d_model) for _ in range(3))
        self.self_attn, self.cross_attn, self.ffn = Attention(cfg), Attention(cfg), FeedForward(cfg)

    def forward(self, x, memory, allowed):
        x = x + self.self_attn(self.ln1(x), causal=True)
        x = x + self.cross_attn(self.ln2(x), memory=memory, allowed=allowed)
        return x + self.ffn(self.ln3(x))


class PrefixBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, prompt, decoder, prefix_allowed):
        # The prompt cannot see decoder tokens; decoder queries see both parts.
        p = self.ln1(prompt)
        d = self.ln1(decoder)
        pk, pv = self.attn.keys_values(p)
        dk, dv = self.attn.keys_values(d)
        prompt = prompt + self.attn.attend(self.attn.query(p), pk, pv, prefix_allowed)
        t = decoder.shape[1]
        triangular = torch.ones(t, t, dtype=torch.bool, device=decoder.device).tril()
        allowed = torch.cat([prefix_allowed.expand(-1, -1, t, -1),
                             triangular[None, None].expand(decoder.shape[0], 1, -1, -1)], dim=-1)
        decoder = decoder + self.attn.attend(self.attn.query(d), torch.cat([pk, dk], dim=2),
                                            torch.cat([pv, dv], dim=2), allowed)
        return prompt + self.ffn(self.ln2(prompt)), decoder + self.ffn(self.ln2(decoder))


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None, role=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        if role is not None:
            x = x + self.segment_emb.weight[role]
        return x


class EncoderDecoderLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.encoder_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.decoder_layers)])
        self.encoder_norm, self.decoder_norm = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.initialize()

    def encode(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt)
        for block in self.encoder:
            x = block(x, allowed)
        return self.encoder_norm(x), allowed

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        memory, allowed = self.encode(prompt, lengths)
        x = self.embed(decoder_ids)
        for block in self.decoder:
            x = block(x, memory, allowed)
        return self.head(self.decoder_norm(x))

    def start_generation(self, prompt, lengths):
        memory, allowed = self.encode(prompt, lengths)
        layers = []
        for block in self.decoder:
            k, v = block.cross_attn.keys_values(memory)
            layers.append({"cross_k": k, "cross_v": v,
                           "k": k[:, :, :0], "v": v[:, :, :0]})
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        position = cache["decoder_position"]
        x = self.embed(ids, offset=torch.full_like(cache["lengths"], position))
        for block, layer in zip(self.decoder, cache["layers"]):
            h = block.ln1(x)
            k, v = block.self_attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            # One new query may see all cached decoder keys, including itself.
            x = x + block.self_attn.attend(block.self_attn.query(h), layer["k"], layer["v"])
            x = x + block.cross_attn.attend(block.cross_attn.query(block.ln2(x)),
                                          layer["cross_k"], layer["cross_v"], cache["allowed"])
            x = x + block.ffn(block.ln3(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.decoder_norm(x))[:, -1], cache


class PrefixLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.segment_emb = nn.Embedding(2, cfg.d_model)
        self.blocks = nn.ModuleList([PrefixBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        p, d = self.embed(prompt, role=0), self.embed(decoder_ids, offset=lengths, role=1)
        for block in self.blocks:
            p, d = block(p, d, allowed)
        return self.head(self.final_norm(d))

    def start_generation(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt, role=0)
        layers = []
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layers.append({"prefix_k": k, "prefix_v": v, "k": k[:, :, :0], "v": v[:, :, :0]})
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"], role=1)
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            allowed = torch.cat([cache["allowed"],
                torch.ones((ids.shape[0], 1, 1, layer["k"].shape[2]), device=ids.device, dtype=torch.bool)], dim=-1)
            x = x + block.attn.attend(block.attn.query(h),
                torch.cat([layer["prefix_k"], layer["k"]], dim=2),
                torch.cat([layer["prefix_v"], layer["v"]], dim=2), allowed)
            x = x + block.ffn(block.ln2(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.final_norm(x))[:, -1], cache


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def build_model(cfg):
    constructors = {"encoder_decoder": EncoderDecoderLM, "prefix": PrefixLM, "efficient": EfficientLM}
    return constructors[cfg.architecture](cfg)


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


def logits_and_labels(lm, x, y, prefix_length):
    if lm.cfg.architecture == "efficient":
        return lm(x), y
    p = int(prefix_length)
    if not 0 < p < x.shape[1]:
        raise ValueError("Need a nonempty prefix and continuation")
    labels = x[:, p:]
    bos = torch.full_like(labels[:, :1], 2)
    decoder = torch.cat([bos, labels[:, :-1]], dim=1)
    return lm(x[:, :p], decoder), labels


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        prefix_length = int(prefixes[0].item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, prefix_length)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/phase1_efficient.py
"""Efficient causal LM: assignment GQA with actual sliding-window attention.

Course Module 2.3 pages 155-156, 167-170 and 180-185.
A3 Q2 supplies the KV-head sharing algorithm. SDPA Boolean True means allowed.
Training uses next-token targets. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "efficient"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4864
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128

    def __post_init__(self):
        if self.architecture != "efficient":
            raise ValueError("This notebook implements the efficient architecture only")
        assert self.d_model % self.n_heads == 0 and self.n_heads % self.n_kv_heads == 0
        assert min(self.vocab_size - 4, self.n_layers, self.d_ff, self.max_seq_len,
                   self.window_size, self.query_chunk_size) > 0
        assert 0 <= self.dropout < 1


def configuration(architecture="efficient"):
    return ModelConfig(architecture=architecture)



def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        return x


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, 0)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


def build_model(cfg):
    return EfficientLM(cfg)


def logits_and_labels(lm, x, y, prefix_length=0):
    # y is the token immediately after each input position in x.
    return lm(x), y


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_data.py
"""CJPE input discovery, official data, saved BPE, document caches and audits."""
import collections
import hashlib
import json
import math
import os
import pickle
import re
import shutil
import time
import zipfile
from pathlib import Path
import numpy as np


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 2**20), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    def pairs(items):
        result = {}
        for k, v in items:
            if k in result:
                raise ValueError("Duplicate JSON key: " + str(k))
            result[k] = v
        return result
    def invalid(x):
        raise ValueError("Nonfinite JSON value: " + x)
    return json.loads(Path(path).read_text(encoding="utf-8-sig"),
                      object_pairs_hook=pairs, parse_constant=invalid)


def safe_error(exc):
    return re.sub(r"hf_[A-Za-z0-9]+", "[redacted]", str(exc))[:3000]


def discover(filename, input_root, supplied=None, hint=None, extraction_root=None):
    """Find one unambiguous file; extract only requested members and companions."""
    base = Path(supplied) if supplied else Path(input_root)
    hints = [hint] if isinstance(hint, str) else list(hint or [])
    if supplied and not base.exists():
        raise FileNotFoundError("Input path does not exist: " + str(base))
    candidates = [base] if base.is_file() and base.name == filename else (
        list(base.rglob(filename)) if base.is_dir() else [])
    if hints and not supplied:
        candidates = [p for p in candidates if any(h.lower() in str(p).lower() for h in hints)]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError("Several " + filename + " files found. Set its exact input path: " +
                         " | ".join(str(p) for p in candidates[:10]))
    archives = [base] if base.is_file() and base.suffix.lower() == ".zip" else (
        sorted(base.rglob("*.zip")) if base.is_dir() else [])
    matches = []
    for archive in archives:
        try:
            with zipfile.ZipFile(archive) as z:
                for info in z.infolist():
                    if not info.is_dir() and Path(info.filename).name == filename:
                        if not hints or supplied or any(h.lower() in (str(archive)+"/"+info.filename).lower() for h in hints):
                            matches.append((archive, info.filename))
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise FileNotFoundError("Need exactly one " + filename + " for " + str(hint or supplied) +
                                "; archive matches=" + str(len(matches)) +
                                ". Attach the original dataset or set its exact path.")
    archive, member = matches[0]
    target_root = Path(extraction_root) / hashlib.sha256(str(archive.resolve()).encode()).hexdigest()[:12]
    parent = Path(member).parent
    with zipfile.ZipFile(archive) as z:
        wanted = [member]
        for companion in ("cache_manifest.json", "sources/phase1_decoder.py", "sources/architecture_models.py"):
            name = str(parent / companion)
            if name in z.namelist():
                wanted.append(name)
        for name in wanted:
            target = (target_root / name).resolve()
            if not target.is_relative_to(target_root.resolve()):
                raise ValueError("Unsafe archive member.")
            info = z.getinfo(name)
            if shutil.disk_usage(target_root.parent if target_root.parent.exists() else Path.cwd()).free < info.file_size + 2**28:
                raise RuntimeError("Insufficient disk for the selected checkpoint.")
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(name) as src, open(str(target) + ".tmp", "wb") as dst:
                shutil.copyfileobj(src, dst, 8 * 2**20)
            Path(str(target) + ".tmp").replace(target)
    return target_root / member


def official_data(root, requested_revision=None, local_input=None):
    """Pin one actual repository commit before downloading only Phase 4 JSON."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    manifest_path = root / "download_manifest.json"
    if manifest_path.exists():
        old = read_json(manifest_path)
        if requested_revision not in (None, "main", old.get("revision")):
            raise ValueError("This working data cache belongs to a different requested revision.")
        if all((root / key).is_file() and sha256(root / key) == value for key, value in old["sha256"].items()):
            print("Reused verified official CJPE files at commit", old["revision"], flush=True)
            return root, old
        raise ValueError("Existing CJPE files differ from their manifest. Use a fresh RUN_ID.")
    relative = {"train": "Train/train.json", "dev": "Train/dev.json", "test": "Test/test.json"}
    if local_input:
        p = Path(local_input)
        if p.is_file() and p.suffix.lower() == ".zip":
            destination = root / "_attached"
            with zipfile.ZipFile(p) as z:
                for split, suffix in relative.items():
                    choices = [n for n in z.namelist() if n.endswith(suffix) and "Phase_4_CJPE" in n]
                    if len(choices) != 1:
                        raise ValueError("Attached CJPE archive needs one Phase_4_CJPE/" + suffix)
                    target = root / suffix
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(choices[0]) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        else:
            for suffix in relative.values():
                choices = [q for q in p.rglob(Path(suffix).name) if str(q).endswith(suffix)]
                if len(choices) != 1:
                    raise ValueError("Choose a directory containing exactly one " + suffix)
                target = root / suffix
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(choices[0], target)
        identity = {"repo_id": "Exploration-Lab/CS-781-Capstone", "revision": "attached_local_copy",
                    "provenance": "user-supplied; byte hashes recorded, remote revision unverified"}
    else:
        from huggingface_hub import HfApi, hf_hub_download
        token = os.environ.get("HF_TOKEN")
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception:
                raise RuntimeError("Enable Internet and grant this notebook access to Kaggle secret HF_TOKEN.") from None
        repo = "Exploration-Lab/CS-781-Capstone"
        try:
            api = HfApi(token=token)
            revision = api.dataset_info(repo, revision=requested_revision or "main").sha
            files = api.list_repo_files(repo_id=repo, repo_type="dataset", revision=revision)
            resolved = {}
            for split, suffix in relative.items():
                exact = "Phase_4_CJPE/" + suffix
                choices = [n for n in files if n == exact]
                if not choices:
                    choices = [n for n in files if n.lower() == exact.lower() or n.lower().endswith("/" + exact.lower())]
                if not choices:
                    choices = [n for n in files if "phase4cjpe" in re.sub(r"[^a-z0-9]", "", n.lower())
                               and n.lower().endswith("/" + suffix.lower())]
                if len(choices) != 1:
                    preview = [n for n in files if "cjpe" in n.lower() or "phase_4" in n.lower()]
                    write_json(root / "phase4_repository_paths.json", preview)
                    raise ValueError("Could not resolve " + exact + "; repository paths saved for inspection.")
                downloaded = hf_hub_download(repo_id=repo, repo_type="dataset", filename=choices[0],
                    revision=revision, token=token, local_dir=str(root / "_download"))
                resolved[split] = choices[0]
                target = root / suffix
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(downloaded, target)
            identity = {"repo_id": repo, "revision": revision, "resolved_files": resolved,
                        "provenance": "official pinned download"}
        except Exception as exc:
            raise RuntimeError("Official CJPE download failed: " + safe_error(exc)) from None
        finally:
            token = None
    identity["sha256"] = {suffix: sha256(root / suffix) for suffix in relative.values()}
    write_json(manifest_path, identity)
    return root, identity


def normalize_records(raw, split, label_field=None):
    """Accept explicit list/dictionary schemas without guessing ACCEPT/REJECT mapping."""
    if isinstance(raw, dict):
        wrappers = [k for k in ("data", "records", "examples", "cases")
                    if isinstance(raw.get(k), list) and "text" not in raw]
        if len(wrappers) == 1:
            raw = raw[wrappers[0]]
    if isinstance(raw, list):
        items = [(None, r) for r in raw]
    elif isinstance(raw, dict) and "id" in raw and "text" in raw:
        items = [(None, raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(split + ": expected a list or an ID-to-record mapping.")
    documents, seen, schema = [], set(), collections.Counter()
    for row_index, (key, value) in enumerate(items):
        if split == "test" and key is not None and isinstance(value, (str, list)):
            value = {"id": key, "text": value}
        if not isinstance(value, dict):
            raise ValueError(split + ": each case must contain text and an ID.")
        case_id = value.get("id", key)
        if not isinstance(case_id, str) or not case_id.strip():
            raise ValueError(split + ": case IDs must be nonempty strings; no ID conversion is applied.")
        if key is not None and key != case_id:
            raise ValueError(split + ": dictionary key and record ID disagree.")
        if case_id in seen:
            schema["rows_repeating_an_original_case_id"] += 1
        seen.add(case_id)
        text = value.get("text")
        if isinstance(text, list) and all(isinstance(x, str) for x in text):
            text = "\n".join(text)
            schema["text_list_joined_with_newlines"] += 1
        elif isinstance(text, str):
            schema["text_string"] += 1
        else:
            raise ValueError(split + "/" + case_id + ": text must be a string or ordered string list.")
        if not text.strip():
            raise ValueError(split + "/" + case_id + ": empty document.")
        gold = None
        if split != "test":
            fields = [label_field] if label_field else [
                k for k in ("gold", "label", "labels", "target", "outcome") if k in value]
            if not fields or any(k not in value for k in fields):
                raise ValueError(split + ": binary label field not found; keys=" + str(sorted(value)))
            labels = []
            for field in fields:
                v = value[field]
                if isinstance(v, list) and len(v) == 1:
                    v = v[0]
                    schema["singleton_label_list_unwrapped"] += 1
                if isinstance(v, str) and v in ("0", "1"):
                    v = int(v)
                    schema["numeric_string_label_converted"] += 1
                if isinstance(v, bool) or not isinstance(v, int) or v not in (0, 1):
                    raise ValueError(split + ": need a scalar binary 0/1 target; no word-label mapping is assumed.")
                labels.append(v)
            if len(set(labels)) != 1:
                raise ValueError(split + ": candidate label fields disagree.")
            gold = labels[0]
            schema["label_field:" + fields[0]] += 1
        schema["records"] += 1
        # Official train/dev may repeat a case ID. Preserve every provided row and raw ID;
        # use a separate identity for caches, local evaluation and teacher-target alignment.
        row_uid = split + ":" + str(row_index).zfill(8) + ":" + case_id
        documents.append({"id": case_id, "row_uid": row_uid, "row_index": row_index,
                          "split": split, "text": text, "gold": gold})
    if not documents:
        raise ValueError(split + ": no cases.")
    return documents, dict(schema)


def load_documents(root, label_field=None):
    documents, schemas = [], {}
    for split, suffix in (("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")):
        part, schemas[split] = normalize_records(read_json(Path(root) / suffix), split, label_field)
        documents.extend(part)
    return documents, schemas


def evaluation_id(document):
    """Local held-out rows are unique; submission IDs stay exactly as supplied."""
    return document["id"] if document["split"] == "test" else document["row_uid"]


def audit_record_ids(documents, output):
    """Audit repeats before BPE; no row removal, label merging, or raw-ID rewriting."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    report = {"record_identity_version": 2,
        "policy": "Preserve all supplied rows and original IDs; use split/row-index/raw-ID row_uid internally.",
        "local_evaluation_unit": "provided development rows",
        "test_output_identity": "unchanged official case ID",
        "submission_blockers": [], "review_warnings": [], "splits": {}}
    examples = []
    for split in ("train", "dev", "test"):
        groups = collections.defaultdict(list)
        for d in documents:
            if d["split"] == split:
                groups[d["id"]].append(d)
        collisions = []
        for case_id, rows in sorted(groups.items()):
            if len(rows) < 2:
                continue
            exact, normalized = set(), collections.defaultdict(list)
            members = []
            for d in rows:
                exact_sha = hashlib.sha256(d["text"].encode()).hexdigest()
                norm_sha = hashlib.sha256(re.sub(r"\s+", " ", d["text"]).strip().casefold().encode()).hexdigest()
                exact.add(exact_sha)
                normalized[norm_sha].append(d)
                members.append({"row_uid": d["row_uid"], "row_index": d["row_index"],
                    "gold": d["gold"], "characters": len(d["text"]), "text_sha256": exact_sha})
            conflicts = [{"normalized_text_sha256": key,
                "row_uids": [d["row_uid"] for d in same],
                "gold_labels": sorted({d["gold"] for d in same if d["gold"] is not None})}
                for key, same in normalized.items()
                if len({d["gold"] for d in same if d["gold"] is not None}) > 1]
            collisions.append({"original_case_id": case_id, "rows": len(rows),
                "distinct_exact_texts": len(exact), "distinct_normalized_texts": len(normalized),
                "label_counts": dict(collections.Counter(str(d["gold"]) for d in rows)),
                "same_normalized_text_conflicting_labels": conflicts, "members": members})
            if len(examples) < 20:
                examples.append({"split": split, "original_case_id": case_id,
                    "members": [{"row_uid": d["row_uid"], "gold": d["gold"],
                        "beginning": d["text"][:200], "end": d["text"][-200:]} for d in rows[:4]]})
        report["splits"][split] = {"provided_rows": sum(map(len, groups.values())),
            "unique_original_case_ids": len(groups), "repeated_original_id_groups": len(collisions),
            "extra_rows_with_repeated_ids": sum(g["rows"]-1 for g in collisions),
            "collision_groups": collisions}
        if split == "test" and collisions:
            report["submission_blockers"].append(
                "Official test IDs repeat; one prediction per official ID is ambiguous. Inspect record_id_audit.json.")
        elif collisions:
            report["review_warnings"].append(split + " contains " + str(len(collisions)) +
                " repeated-ID groups; all rows retained and reported separately.")
    write_json(output/"record_id_audit.json", report)
    write_json(output/"repeated_id_examples.json", examples)
    return report


def load_tokenizer(path, expected_hash=None):
    from a1_tokenizer import A1BPETokenizer
    path = Path(path)
    manifest = read_json(path.parent / "cache_manifest.json")
    digest = sha256(path)
    if digest != manifest["tokenizer_state_sha256"] or (expected_hash and digest != expected_hash):
        raise ValueError("Use the original prep-2 tokenizer and its matching cache_manifest.json.")
    with open(path, "rb") as f:
        saved = pickle.load(f)  # Only the user's own Phase 1 preparation is loaded.
    if saved["signature"] != manifest["tokenizer_signature"]:
        raise ValueError("Tokenizer signature mismatch.")
    tokenizer = A1BPETokenizer()
    tokenizer.__dict__.update(saved["state"])
    if tokenizer.vocab[:4] != [b"<pad>", b"<unk>", b"<s>", b"</s>"]:
        raise ValueError("Unexpected reserved token IDs.")
    probes = ["A court allowed the appeal.", "Dismissed.\nThe judgment follows.", "धारा १२"]
    for sample in probes:
        ids = tokenizer.encode(sample)
        if tokenizer.decode(ids) != sample or not ids or min(ids) < 4 or max(ids) >= len(tokenizer.vocab):
            raise ValueError("Saved BPE round-trip or token-ID range check failed.")
    return tokenizer, saved["signature"], manifest


def tokenize_documents(documents, tokenizer, cache_dir, identity):
    """Per-case atomic caches survive interruption; BPE is never retrained."""
    from functools import lru_cache
    folder = Path(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    @lru_cache(maxsize=250000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    last, started = time.monotonic(), time.monotonic()
    cache_files = []
    for i, doc in enumerate(documents):
        text_sha = hashlib.sha256(doc["text"].encode()).hexdigest()
        key = hashlib.sha256(json.dumps([fingerprint, doc["row_uid"], text_sha]).encode()).hexdigest()
        path = folder / (key + ".npz")
        if path.exists():
            with np.load(path, allow_pickle=False) as data:
                if str(data["identity"]) != key:
                    raise ValueError("Token-cache identity mismatch.")
                ids = data["ids"].copy()
        else:
            ids = np.asarray([t for piece in re.findall(r"\S+|\s+", doc["text"])
                              for t in piece_ids(piece)], dtype=np.int32)
            with open(str(path) + ".tmp", "wb") as f:
                np.savez_compressed(f, ids=ids, identity=np.asarray(key))
            Path(str(path) + ".tmp").replace(path)
        if ids.ndim != 1 or not len(ids) or ids.min() < 4 or ids.max() >= len(tokenizer.vocab):
            raise ValueError("Invalid cached content token IDs.")
        doc["token_ids"] = ids
        doc["text_sha256"] = text_sha
        doc["token_file"] = path.name
        cache_files.append({"id": doc["id"], "row_uid": doc["row_uid"], "row_index": doc["row_index"],
                            "split": doc["split"], "gold": doc["gold"],
                            "token_file": path.name, "text_sha256": text_sha, "content_tokens": len(ids)})
        if i == 0 or time.monotonic() - last >= 25 or i + 1 == len(documents):
            print("Tokenized/reused", i + 1, "/", len(documents), "cases |",
                  round((time.monotonic() - started) / 60, 1), "minutes", flush=True)
            last = time.monotonic()
    write_json(folder / "index.json", {"record_identity_version": 2, "fingerprint": fingerprint,
                                      "identity": identity, "cases": cache_files})
    return {"fingerprint": fingerprint, "cases": len(documents),
            "seconds": time.monotonic() - started, "unique_cached_pieces": piece_ids.cache_info().currsize}


def chunk_spans(n_tokens, total_tokens=512, overlap=64):
    content = total_tokens - 2
    if n_tokens < 1 or content < 1 or not 0 <= overlap < content:
        raise ValueError("Invalid chunk geometry.")
    spans = []
    start = 0
    while True:
        stop = min(start + content, n_tokens)
        spans.append((start, stop))
        if stop == n_tokens:
            return spans
        start += content - overlap


def selected_spans(n_tokens, total_tokens, overlap, max_chunks, policy="uniform"):
    spans = chunk_spans(n_tokens, total_tokens, overlap)
    if max_chunks < 1:
        raise ValueError("max_chunks must be positive.")
    if len(spans) <= max_chunks:
        return spans
    if policy == "uniform":
        indices = np.linspace(0, len(spans) - 1, max_chunks).round().astype(int).tolist()
    elif policy == "head":
        indices = list(range(max_chunks))
    elif policy == "tail":
        indices = list(range(len(spans) - max_chunks, len(spans)))
    elif policy == "head_tail":
        left = (max_chunks + 1) // 2
        indices = list(range(left)) + list(range(len(spans) - (max_chunks - left), len(spans)))
    else:
        raise ValueError("Unknown chunk selection policy.")
    if len(set(indices)) != max_chunks:
        raise AssertionError("Repeated selected chunk.")
    return [spans[i] for i in sorted(indices)]


def coverage(n_tokens, spans):
    covered, end = 0, 0
    for start, stop in spans:
        covered += max(0, stop - max(start, end))
        end = max(end, stop)
    return covered / n_tokens


def audit_documents(documents, schemas, output):
    import pandas as pd
    from sklearn.metrics import f1_score, precision_score, recall_score
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    rows, summary = [], {"schemas": schemas, "label_meaning": "Numeric mapping preserved; ACCEPT/REJECT meaning not assumed."}
    for split in ("train", "dev", "test"):
        part = [d for d in documents if d["split"] == split]
        counts = collections.Counter(d["gold"] for d in part) if split != "test" else {}
        summary[split] = {"cases": len(part), "labels": {str(k): v for k, v in counts.items()},
            "unique_original_case_ids": len({d["id"] for d in part}),
            "metric_and_sampling_unit": "provided rows; row_uid identifies each example",
            "content_tokens": {str(q): float(np.percentile([len(d["token_ids"]) for d in part], q))
                               for q in (0, 25, 50, 75, 90, 95, 99, 100)}}
        for d in part:
            rows.append({"split": split, "id": d["id"], "row_uid": d["row_uid"],
                "row_index": d["row_index"], "gold": d["gold"], "characters": len(d["text"]),
                "words": len(d["text"].split()), "content_tokens": len(d["token_ids"]),
                "text_sha256": d["text_sha256"]})
    problems, overlap = [], []
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        pa = [d for d in documents if d["split"] == a]
        pb = [d for d in documents if d["split"] == b]
        ids = sorted({d["id"] for d in pa} & {d["id"] for d in pb})
        # Whitespace-normalized duplicate detection catches formatting-only copies.
        def digest(d):
            return hashlib.sha256(re.sub(r"\s+", " ", d["text"]).strip().casefold().encode()).hexdigest()
        ma, mb = collections.defaultdict(list), collections.defaultdict(list)
        for d in pa:
            ma[digest(d)].append({"id": d["id"], "row_uid": d["row_uid"]})
        for d in pb:
            mb[digest(d)].append({"id": d["id"], "row_uid": d["row_uid"]})
        duplicate_pairs = [{"a_ids": ma[k], "b_ids": mb[k]} for k in sorted(set(ma) & set(mb))]
        overlap.append({"a": a, "b": b, "shared_ids": ids, "normalized_text_duplicates": duplicate_pairs})
        if ids or duplicate_pairs:
            problems.append("Overlap detected between " + a + " and " + b + "; inspect before training.")
    for split in ("train", "dev"):
        if set(summary[split]["labels"]) != {"0", "1"}:
            problems.append(split + " does not contain both binary classes.")
    summary["cross_split_overlap"] = overlap
    summary["training_blockers"] = problems
    within = {}
    for split in ("train", "dev", "test"):
        groups = collections.defaultdict(list)
        for d in documents:
            if d["split"] == split:
                groups[d["text_sha256"]].append({"id": d["id"], "row_uid": d["row_uid"], "gold": d["gold"]})
        within[split] = [ids for ids in groups.values() if len(ids) > 1]
    summary["within_split_exact_duplicate_groups"] = within
    majority = int(max(collections.Counter(d["gold"] for d in documents if d["split"] == "train").items(),
                       key=lambda pair: (pair[1], -pair[0]))[0])
    gold = [d["gold"] for d in documents if d["split"] == "dev"]
    summary["majority_label_from_train"] = majority
    summary["dev_constant_baselines"] = {}
    for label in (0, 1):
        pred = [label] * len(gold)
        summary["dev_constant_baselines"][str(label)] = {
            "macro_f1": float(f1_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
            "precision_1": float(precision_score(gold, pred, zero_division=0)),
            "recall_1": float(recall_score(gold, pred, zero_division=0))}
    examples = []
    for split in ("train", "dev"):
        for label in (0, 1):
            candidates = [d for d in documents if d["split"] == split and d["gold"] == label]
            if candidates:
                d = sorted(candidates, key=lambda d: len(d["token_ids"]))[len(candidates) // 2]
                middle = len(d["text"]) // 2
                examples.append({"id": d["id"], "row_uid": d["row_uid"], "split": split, "gold": label,
                    "beginning": d["text"][:400], "middle": d["text"][middle:middle+400], "end": d["text"][-400:]})
    write_json(output / "data_audit.json", summary)
    write_json(output / "train_dev_examples.json", examples)
    pd.DataFrame(rows).to_csv(output / "case_statistics.csv", index=False)
    return summary


def audit_chunk_policies(documents, output, candidates):
    import pandas as pd
    rows = []
    for setting in candidates:
        for split in ("train", "dev", "test"):
            docs = [d for d in documents if d["split"] == split]
            spans = [selected_spans(len(d["token_ids"]), **setting) for d in docs]
            covered = [coverage(len(d["token_ids"]), s) for d, s in zip(docs, spans)]
            rows.append(dict(setting, split=split, cases=len(docs),
                mean_chunks=float(np.mean([len(s) for s in spans])),
                selected_chunks=sum(map(len, spans)), mean_content_coverage=float(np.mean(covered)),
                p10_content_coverage=float(np.percentile(covered, 10)),
                full_coverage_fraction=float(np.mean([v >= 1 - 1e-12 for v in covered])),
                total_encoded_tokens=sum(stop-start+2 for ss in spans for start, stop in ss)))
    pd.DataFrame(rows).to_csv(Path(output) / "chunk_policy_comparison.csv", index=False)
    return rows


def collate_cases(documents, total_tokens=512, overlap=64, max_chunks=8, policy="uniform"):
    import torch
    selected = [selected_spans(len(d["token_ids"]), total_tokens, overlap, max_chunks, policy)
                for d in documents]
    k = max(map(len, selected))
    width = max(stop-start+2 for spans in selected for start, stop in spans)
    ids = torch.zeros((len(documents), k, width), dtype=torch.long)
    lengths = torch.zeros((len(documents), k), dtype=torch.long)
    positions = torch.zeros((len(documents), k, 2), dtype=torch.float32)
    for b, (d, spans) in enumerate(zip(documents, selected)):
        for j, (start, stop) in enumerate(spans):
            tokens = [2] + [int(v) for v in d["token_ids"][start:stop]] + [3]
            ids[b, j, :len(tokens)] = torch.tensor(tokens)
            lengths[b, j] = len(tokens)
            positions[b, j] = torch.tensor([start / len(d["token_ids"]), stop / len(d["token_ids"])])
    labels = torch.tensor([d["gold"] if d["gold"] is not None else -100 for d in documents])
    return {"ids": ids, "lengths": lengths, "positions": positions}, labels


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_models.py
"""Phase 1-derived case classifiers. No external pretrained models or training loop."""
import contextlib
import copy
import math
from dataclasses import replace
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from phase1_decoder import DecoderLM, LMConfig
import phase1_shared as shared
import phase1_efficient as efficient


def mask_for(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


def mean_content(states, lengths):
    positions = torch.arange(states.shape[1], device=states.device)[None, :]
    valid = (positions > 0) & (positions < lengths[:, None] - 1)
    count = valid.sum(1)
    if not bool((count > 0).all()):
        raise ValueError("Each chunk needs at least one content token.")
    return (states.float() * valid[:, :, None]).sum(1) / count[:, None]


def chunk_states(lm, architecture, ids, lengths, use_checkpoint=False):
    """Return token-context states and a full-context query/last-content readout."""
    if ids.ndim != 2 or lengths.shape != (len(ids),):
        raise ValueError("Expected IDs [chunks,tokens] and lengths [chunks].")
    if not bool(((lengths >= 3) & (lengths <= ids.shape[1])).all()):
        raise ValueError("Invalid nonpadding chunk length.")
    reserve = int(architecture in ("prefix", "encoder_decoder"))
    if ids.shape[1] + reserve > lm.cfg.max_seq_len:
        raise ValueError("Chunk plus classification query exceeds the Phase 1 context budget.")
    allowed = mask_for(lengths, ids.shape[1])[:, None, None, :]
    clean = ids.masked_fill(~allowed[:, 0, 0, :], 0)
    call = lambda block, *args: checkpoint(block, *args, use_reentrant=False,
                                            preserve_rng_state=True) if use_checkpoint else block(*args)
    if architecture == "decoder":
        x = lm.tok_emb(clean) * lm.cfg.embedding_scale + lm.pos_emb[:ids.shape[1]][None]
        for block in lm.blocks:
            x = call(block, x)
        states = lm.ln_f(x)
        readout = states[torch.arange(len(ids), device=ids.device), lengths - 2]
    elif architecture == "efficient":
        x = lm.embed(clean)
        for block in lm.blocks:
            x = call(block, x, lengths)
        states = lm.final_norm(x)
        readout = states[torch.arange(len(ids), device=ids.device), lengths - 2]
    elif architecture == "encoder_decoder":
        memory = lm.embed(clean)
        for block in lm.encoder:
            memory = call(block, memory, allowed)
        states = lm.encoder_norm(memory)
        query = torch.full((len(ids), 1), 2, dtype=torch.long, device=ids.device)
        x = lm.embed(query)
        for block in lm.decoder:
            x = call(block, x, states, allowed)
        readout = lm.decoder_norm(x)[:, 0]
    elif architecture == "prefix":
        p = lm.embed(clean, role=0)
        query = torch.full((len(ids), 1), 2, dtype=torch.long, device=ids.device)
        d = lm.embed(query, offset=lengths, role=1)
        for block in lm.blocks:
            p, d = call(block, p, d, allowed)
        states, readout = lm.final_norm(p), lm.final_norm(d)[:, 0]
    else:
        raise ValueError("Unknown architecture.")
    return states, readout


class Bottleneck(nn.Module):
    def __init__(self, width, bottleneck, dropout):
        super().__init__()
        self.down = nn.Linear(width, bottleneck)
        self.up = nn.Linear(bottleneck, width)
        self.dropout = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.dropout(self.up(F.gelu(self.down(x))))


class AdaptedOutput(nn.Module):
    """Adapter after a sublayer output, before its outer residual addition."""
    def __init__(self, original, width, bottleneck, dropout):
        super().__init__()
        self.original = original
        self.adapter = Bottleneck(width, bottleneck, dropout)

    def forward(self, *args, **kwargs):
        return self.adapter(self.original(*args, **kwargs))


class LoRMAPlusLinear(nn.Module):
    """Output-side (I + alpha/r BA) W0 x + original bias."""
    def __init__(self, original, rank, alpha, dropout):
        super().__init__()
        if not isinstance(original, nn.Linear) or not 0 < rank <= min(original.weight.shape):
            raise ValueError("Invalid LoRMA+ linear target or rank.")
        self.original = original
        self.lorma_A = nn.Parameter(original.weight.new_empty(rank, original.out_features))
        self.lorma_B = nn.Parameter(original.weight.new_zeros(original.out_features, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)
        self.scaling = alpha / rank
        for p in original.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        base = self.original(x)
        projected = base if self.original.bias is None else base - self.original.bias.to(base.dtype)
        return base + self.scaling * F.linear(F.linear(self.dropout(projected), self.lorma_A), self.lorma_B)


class DocumentBiGRU(nn.Module):
    """Small GRU written with tensor operations, applied only to valid chunks."""
    def __init__(self, width, hidden):
        super().__init__()
        self.hidden = hidden
        self.input = nn.ModuleList([nn.Linear(width, 3 * hidden) for _ in range(2)])
        self.recurrent = nn.ModuleList([nn.Linear(hidden, 3 * hidden) for _ in range(2)])

    def forward(self, x):
        outputs = []
        for direction in range(2):
            hidden = x.new_zeros(self.hidden)
            transformed = self.input[direction](x)
            states = [None] * len(x)
            order = range(len(x)) if direction == 0 else range(len(x)-1, -1, -1)
            for i in order:
                xr, xz, xn = transformed[i].chunk(3, -1)
                hr, hz, hn = self.recurrent[direction](hidden).chunk(3, -1)
                reset, update = torch.sigmoid(xr+hr), torch.sigmoid(xz+hz)
                candidate = torch.tanh(xn + reset*hn)
                hidden = (1-update)*candidate + update*hidden
                states[i] = hidden
            outputs.append(torch.stack(states))
        return torch.cat(outputs, -1)


def compact_efficient(lm, layers):
    if not layers or len(set(layers)) != len(layers) or list(layers) != sorted(layers):
        raise ValueError("Student layers must be distinct, ascending zero-based indices.")
    if min(layers) < 0 or max(layers) >= len(lm.blocks) or len(layers) >= len(lm.blocks):
        raise ValueError("The student must retain fewer than all original efficient blocks.")
    result = efficient.EfficientLM(replace(lm.cfg, n_layers=len(layers)))
    result.tok_emb.load_state_dict(lm.tok_emb.state_dict(), strict=True)
    result.final_norm.load_state_dict(lm.final_norm.state_dict(), strict=True)
    for new, old in enumerate(layers):
        result.blocks[new].load_state_dict(lm.blocks[old].state_dict(), strict=True)
    if result.tok_emb.weight.data_ptr() == lm.tok_emb.weight.data_ptr():
        raise AssertionError("Student and original must not share mutable parameter storage.")
    return result


def configure_dropout(lm, probability):
    lm.cfg = replace(lm.cfg, dropout=float(probability))
    for module in lm.modules():
        if isinstance(module, nn.Dropout):
            module.p = float(probability)
        elif isinstance(module, (shared.Attention, efficient.Attention)):
            module.dropout = float(probability)


class CaseClassifier(nn.Module):
    def __init__(self, lm, architecture, settings):
        super().__init__()
        self.lm, self.architecture, self.settings = lm, architecture, dict(settings)
        self.use_amp = True
        self.gradient_checkpointing = bool(settings["gradient_checkpointing"])
        self.chunk_microbatch = int(settings["chunk_microbatch"])
        configure_dropout(lm, settings["backbone_dropout"])
        for p in lm.parameters():
            p.requires_grad_(False)
        self.adaptation_targets = []
        if architecture == "decoder":
            upper = settings["decoder_upper_layers"]
            if not 1 <= upper <= len(lm.blocks):
                raise ValueError("Invalid partial fine-tuning depth.")
            for i in range(len(lm.blocks)-upper, len(lm.blocks)):
                for p in lm.blocks[i].parameters():
                    p.requires_grad_(True)
                self.adaptation_targets.append("blocks." + str(i))
            for p in lm.ln_f.parameters():
                p.requires_grad_(True)
        elif architecture == "encoder_decoder":
            for group, blocks in (("encoder", lm.encoder), ("decoder", lm.decoder)):
                for i, block in enumerate(blocks):
                    names = ("attn", "ffn") if group == "encoder" else ("self_attn", "cross_attn", "ffn")
                    for name in names:
                        setattr(block, name, AdaptedOutput(getattr(block, name), lm.cfg.d_model,
                            settings["adapter_bottleneck"], settings["adapter_dropout"]))
                        self.adaptation_targets.append(group + "." + str(i) + "." + name)
            if settings["adapter_train_layer_norm"]:
                for module in lm.modules():
                    if isinstance(module, nn.LayerNorm):
                        for p in module.parameters():
                            p.requires_grad_(True)
        elif architecture == "prefix":
            for i, block in enumerate(lm.blocks):
                for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
                    setattr(block.attn, name, LoRMAPlusLinear(getattr(block.attn, name),
                        settings["lorma_rank"], settings["lorma_alpha"], settings["lorma_dropout"]))
                    self.adaptation_targets.append("blocks." + str(i) + ".attn." + name)
                for j in (0, 2):
                    block.ffn.net[j] = LoRMAPlusLinear(block.ffn.net[j], settings["lorma_rank"],
                        settings["lorma_alpha"], settings["lorma_dropout"])
                    self.adaptation_targets.append("blocks." + str(i) + ".ffn.net." + str(j))
        elif architecture == "efficient":
            for p in lm.parameters():
                p.requires_grad_(True)
        else:
            raise ValueError("Unknown classifier architecture.")
        # Seed the task head separately: matched gold and KD students get identical heads.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(settings["head_seed"])
            self.projection = nn.Linear(2*lm.cfg.d_model + 2, settings["projection_dim"])
            self.context = DocumentBiGRU(settings["projection_dim"], settings["context_hidden"])
            width = 2*settings["context_hidden"]
            self.attention = nn.Sequential(nn.Linear(width, settings["attention_hidden"]), nn.Tanh(),
                                          nn.Linear(settings["attention_hidden"], 1, bias=False))
            self.classifier = nn.Linear(width, 2)
        self.dropout = nn.Dropout(settings["head_dropout"])

    def chunk_vectors(self, ids, lengths):
        states, readout = chunk_states(self.lm, self.architecture, ids, lengths,
            self.training and self.gradient_checkpointing)
        return torch.cat([mean_content(states, lengths), readout.float()], -1)

    def forward(self, ids, lengths, positions):
        if ids.ndim != 3 or lengths.shape != ids.shape[:2] or positions.shape != (*ids.shape[:2], 2):
            raise ValueError("Expected one chunk sequence per case.")
        logits, weights = [], []
        with torch.autocast("cuda", dtype=torch.float16, enabled=ids.is_cuda and self.use_amp):
            for b in range(len(ids)):
                active = lengths[b] > 0
                n = int(active.sum())
                if n == 0 or not bool(active[:n].all()) or bool(active[n:].any()):
                    raise ValueError("Valid chunks must form one nonempty prefix.")
                pieces = []
                for start in range(0, n, self.chunk_microbatch):
                    stop = min(start+self.chunk_microbatch, n)
                    width = int(lengths[b, start:stop].max())
                    pieces.append(self.chunk_vectors(ids[b, start:stop, :width], lengths[b, start:stop]))
                chunk_vectors = torch.cat(pieces)
                projected = torch.tanh(self.projection(torch.cat([chunk_vectors, positions[b, :n].float()], -1)))
                contextual = self.context(self.dropout(projected))
                alpha = torch.softmax(self.attention(contextual).squeeze(-1).float(), 0)
                document = (contextual.float()*alpha[:, None]).sum(0)
                logits.append(self.classifier(self.dropout(document)))
                weights.append(F.pad(alpha, (0, ids.shape[1]-n)))
        return {"logits": torch.stack(logits).float(), "chunk_attention": torch.stack(weights),
                "device_ids": ids.new_full((len(ids),), ids.device.index if ids.is_cuda else -1)}


def kd_objective(student_logits, labels, teacher_logits=None, temperature=2.0, kd_weight=0.5):
    if temperature <= 0 or not 0 <= kd_weight <= 1:
        raise ValueError("Invalid distillation temperature or mixing weight.")
    if bool(((labels < 0) | (labels > 1)).any()):
        raise ValueError("Training loss requires labelled training cases.")
    gold = F.cross_entropy(student_logits.float(), labels)
    kd = gold*0
    if teacher_logits is not None:
        if teacher_logits.shape != student_logits.shape:
            raise ValueError("Teacher/student class-order or batch shape mismatch.")
        q = torch.softmax(teacher_logits.detach().float()/temperature, -1)
        kd = F.kl_div(F.log_softmax(student_logits.float()/temperature, -1), q,
                      reduction="batchmean") * temperature**2
    elif kd_weight:
        raise ValueError("KD training requires genuine trained-teacher logits.")
    return (1-kd_weight)*gold + kd_weight*kd, {"gold_ce": gold.detach(), "kd": kd.detach()}


def expected_trainability(model, name):
    if not name.startswith("lm."):
        return True
    if model.architecture == "efficient":
        return True
    if model.architecture == "decoder":
        upper = len(model.lm.blocks)-model.settings["decoder_upper_layers"]
        return name.startswith("lm.ln_f.") or any(name.startswith("lm.blocks."+str(i)+".")
                                                  for i in range(upper, len(model.lm.blocks)))
    if model.architecture == "prefix":
        return name.endswith((".lorma_A", ".lorma_B"))
    if model.architecture == "encoder_decoder":
        if ".adapter." in name:
            return True
        if model.settings["adapter_train_layer_norm"]:
            return any(name.startswith(prefix) for prefix in (
                "lm.encoder_norm.", "lm.decoder_norm.")) or any(
                ".ln"+str(i)+"." in name for i in (1, 2, 3))
        return False
    raise ValueError("Unknown architecture.")


def trainability_audit(model):
    counts = {"total_parameters": 0, "trainable_parameters": 0, "backbone_trainable": 0, "task_trainable": 0}
    for name, p in model.named_parameters():
        if p.requires_grad != expected_trainability(model, name):
            raise AssertionError("Unexpected trainability: " + name)
        counts["total_parameters"] += p.numel()
        if p.requires_grad:
            counts["trainable_parameters"] += p.numel()
            counts["backbone_trainable" if name.startswith("lm.") else "task_trainable"] += p.numel()
    counts["adaptation_targets"] = model.adaptation_targets
    counts["parameter_bytes_float32"] = counts["total_parameters"]*4
    return counts


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_checks.py
"""Bounded correctness, temporary learning, two-device and resource probes."""
import copy
import gc
import hashlib
import io
import math
import random
import time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from cjpe_data import sha256, write_json, safe_error, collate_cases, chunk_spans, selected_spans, coverage
from cjpe_models import (CaseClassifier, Bottleneck, LoRMAPlusLinear, DocumentBiGRU,
    compact_efficient, chunk_states, kd_objective, trainability_audit)
from phase1_decoder import DecoderLM, LMConfig
import phase1_shared as shared
import phase1_efficient as efficient


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def tensor_hash(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()


def trainable_hash(model):
    h = hashlib.sha256()
    for name, p in model.named_parameters():
        if p.requires_grad:
            h.update(name.encode())
            h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def load_base(path, architecture, signature):
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if saved["tokenizer_signature"] != signature:
        raise ValueError("Phase 1 checkpoint and prep-2 tokenizer signatures differ.")
    cfg = saved["model_config"]
    if architecture == "decoder":
        lm = DecoderLM(LMConfig(**cfg))
    elif architecture == "efficient":
        lm = efficient.EfficientLM(efficient.ModelConfig(**cfg))
    else:
        config = shared.ModelConfig(**cfg)
        if config.architecture != architecture:
            raise ValueError("Incorrect checkpoint architecture.")
        lm = shared.EncoderDecoderLM(config) if architecture == "encoder_decoder" else shared.PrefixLM(config)
    if not torch.equal(saved["model_state"]["tok_emb.weight"], saved["model_state"]["head.weight"]):
        raise ValueError("The checkpoint's tied embedding/LM head values disagree.")
    lm.load_state_dict(saved["model_state"], strict=True)
    if lm.head.weight is not lm.tok_emb.weight:
        raise AssertionError("Original embedding/output weight tying was lost.")
    report = {"model_config": cfg, "checkpoint_keys": sorted(saved),
              "original_parameters": sum(p.numel() for p in lm.parameters()),
              "strict_state_dict_load": True, "tied_embedding_preserved": True,
              "tokenizer_signature_matches": True}
    del saved
    gc.collect()
    return lm, report


def native_readout_check(lm, architecture, device):
    lm = lm.to(device).eval()
    width = min(17, lm.cfg.max_seq_len-1)
    ids = torch.randint(4, lm.cfg.vocab_size, (2, width), device=device)
    ids[:, 0], ids[:, -1] = 2, 3
    lengths = torch.full((2,), width, device=device, dtype=torch.long)
    query = torch.full((2, 1), 2, device=device, dtype=torch.long)
    with torch.no_grad():
        _, hidden = chunk_states(lm, architecture, ids, lengths)
        if architecture in ("prefix", "encoder_decoder"):
            expected = lm(ids, query, lengths)[:, 0]
        else:
            expected = lm(ids)[:, -2]
        actual = lm.head(hidden)
    torch.testing.assert_close(actual, expected, atol=2e-5, rtol=2e-4)
    return {"status": "passed", "max_abs_logit_difference": float((actual-expected).abs().max()),
            "query": "single BOS token through original decoder stream" if architecture in
            ("prefix", "encoder_decoder") else "last content token"}


def helper_checks():
    seed_all(19)
    for n in (1, 17, 30, 31, 32, 100, 1001):
        all_spans = chunk_spans(n, 32, 7)
        assert coverage(n, all_spans) == 1.0
        assert all(0 <= a < b <= n and b-a <= 30 for a, b in all_spans)
        for cap in (1, 2, 4, 8):
            for policy in ("uniform", "head", "tail", "head_tail"):
                spans = selected_spans(n, 32, 7, cap, policy)
                assert len(spans) <= cap and len(set(spans)) == len(spans)
                assert 0 < coverage(n, spans) <= 1.0
    original = nn.Linear(5, 7, bias=True).double()
    adapter = LoRMAPlusLinear(original, 2, 4, 0).double()
    x = torch.randn(3, 5, dtype=torch.float64, requires_grad=True)
    torch.testing.assert_close(adapter(x), original(x), rtol=0, atol=0)
    with torch.no_grad():
        adapter.lorma_B.normal_(std=0.05)
    matrix = original.weight + adapter.scaling*adapter.lorma_B@adapter.lorma_A@original.weight
    torch.testing.assert_close(adapter(x), F.linear(x, matrix, original.bias), atol=1e-12, rtol=1e-12)
    assert torch.autograd.gradcheck(adapter, (x,), eps=1e-6, atol=1e-5)
    bottleneck = Bottleneck(7, 3, 0).double()
    y = torch.randn(4, 7, dtype=torch.float64)
    torch.testing.assert_close(bottleneck(y), y, atol=0, rtol=0)
    # Compare the manually implemented GRU against an independent library reference.
    manual = DocumentBiGRU(5, 3).double()
    reference = nn.GRU(5, 3, bidirectional=True).double()
    with torch.no_grad():
        for direction, suffix in ((0, ""), (1, "_reverse")):
            getattr(reference, "weight_ih_l0"+suffix).copy_(manual.input[direction].weight)
            getattr(reference, "bias_ih_l0"+suffix).copy_(manual.input[direction].bias)
            getattr(reference, "weight_hh_l0"+suffix).copy_(manual.recurrent[direction].weight)
            getattr(reference, "bias_hh_l0"+suffix).copy_(manual.recurrent[direction].bias)
    seq = torch.randn(6, 5, dtype=torch.float64)
    torch.testing.assert_close(manual(seq), reference(seq[:, None])[0][:, 0], atol=1e-12, rtol=1e-12)
    logits = torch.tensor([[0.3, -0.2], [-0.7, 1.0]], requires_grad=True)
    labels = torch.tensor([0, 1])
    total, pieces = kd_objective(logits, labels, logits.detach(), temperature=2, kd_weight=.5)
    assert abs(float(pieces["kd"])) < 1e-6
    gold, _ = kd_objective(logits, labels, kd_weight=0)
    torch.testing.assert_close(gold, F.cross_entropy(logits, labels))
    total.backward()
    assert torch.isfinite(logits.grad).all()
    try:
        kd_objective(logits, labels)
    except ValueError:
        pass
    else:
        raise AssertionError("KD without teacher targets must be rejected.")
    return {"status": "passed", "checks": ["chunk bounds and full-coverage geometry",
        "LoRMA+ identity, rectangular biased matrix equivalence and numerical input gradients",
        "zero-initialized bottleneck identity", "manual BiGRU versus nn.GRU",
        "binary CE/KD values, gradients and missing-teacher guard"],
        "synthetic_kd_targets_are_not_valid_for_real_training": True}


def batch_to(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def padding_check(model, batch):
    model.eval()
    old = model.use_amp
    model.use_amp = False
    with torch.no_grad():
        original = model(**batch)
        changed = {k: v.clone() for k, v in batch.items()}
        valid_tokens = torch.arange(batch["ids"].shape[2], device=batch["ids"].device)[None, None] < batch["lengths"][:, :, None]
        changed["ids"][~valid_tokens] = 5
        altered = model(**changed)
        extended = {
            "ids": F.pad(batch["ids"], (0, 3, 0, 2), value=9),
            "lengths": F.pad(batch["lengths"], (0, 2)),
            "positions": F.pad(batch["positions"], (0, 0, 0, 2), value=0.7)}
        padded = model(**extended)
    torch.testing.assert_close(original["logits"], altered["logits"], atol=3e-5, rtol=3e-4)
    torch.testing.assert_close(original["logits"], padded["logits"], atol=3e-5, rtol=3e-4)
    torch.testing.assert_close(original["chunk_attention"].sum(1),
                               torch.ones(len(batch["ids"]), device=batch["ids"].device))
    assert bool((padded["chunk_attention"][:, -2:] == 0).all())
    assert torch.isfinite(original["logits"]).all()
    model.use_amp = old
    return {"status": "passed", "changed_pad_tokens_and_extra_pad_chunks_ignored": True,
            "attention_weights_normalize_over_real_chunks": True}


def two_gpu_check(model, batch, labels):
    if not batch["ids"].is_cuda or torch.cuda.device_count() < 2:
        return {"status": "not_run", "reason": "Two CUDA devices unavailable."}
    model.eval()
    old = model.use_amp
    model.use_amp = False
    model.zero_grad(set_to_none=True)
    serial = model(**batch)["logits"]
    F.cross_entropy(serial, labels).backward()
    gradients = {n: p.grad.detach().cpu().clone() for n, p in model.named_parameters()
                 if p.requires_grad and p.grad is not None}
    model.zero_grad(set_to_none=True)
    parallel = nn.DataParallel(model, device_ids=[0, 1])
    result = parallel(**batch)
    assert set(result["device_ids"].cpu().tolist()) == {0, 1}
    F.cross_entropy(result["logits"], labels).backward()
    torch.testing.assert_close(serial, result["logits"], rtol=1e-3, atol=3e-5)
    maximum = 0.0
    for name, p in model.named_parameters():
        if name in gradients:
            if p.grad is None:
                raise AssertionError("Missing parallel gradient: " + name)
            actual = p.grad.detach().cpu()
            torch.testing.assert_close(actual, gradients[name], rtol=2e-3, atol=5e-5)
            maximum = max(maximum, float((actual-gradients[name]).abs().max()))
        elif p.requires_grad and p.grad is not None and float(p.grad.abs().max()) > 0:
            raise AssertionError("Unexpected additional parallel gradient: " + name)
    model.zero_grad(set_to_none=True)
    model.use_amp = old
    del parallel, gradients
    return {"status": "passed", "devices_observed": [0, 1],
            "max_abs_gradient_difference": maximum, "dropout_disabled_for_comparison": True}


def temporary_learning(model, batch, labels, steps):
    device = batch["ids"].device
    model.use_amp = device.type == "cuda"
    # Isolate gradient/learning mechanics from stochastic regularization in this tiny test.
    dropout_backup = [(m, "p", m.p) for m in model.modules() if isinstance(m, nn.Dropout)]
    dropout_backup += [(m, "dropout", m.dropout) for m in model.modules()
                       if isinstance(getattr(m, "dropout", None), float)]
    for module, attribute, _ in dropout_backup:
        setattr(module, attribute, 0.0)
    def evaluation_loss():
        model.eval()
        with torch.no_grad():
            return float(F.cross_entropy(model(**batch)["logits"], labels))
    before = evaluation_loss()
    optimizer = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("lm.")],
         "lr": 1e-5 if model.architecture in ("decoder", "efficient") else 1e-4},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("lm.")],
         "lr": 1e-3}], weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    rows, started = [], time.monotonic()
    gradient_audit = {}
    for step in range(steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(**batch)["logits"]
        # KD wiring is checked in the student too; these synthetic targets never persist.
        teacher = torch.stack([1.5*(labels == 0), 1.5*(labels == 1)], -1).float() if model.architecture == "efficient" else None
        loss, _ = kd_objective(logits, labels, teacher, kd_weight=.3 if teacher is not None else 0)
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite temporary training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):
            raise ValueError("Nonfinite or missing temporary gradients.")
        frozen_bad = [n for n, p in model.named_parameters() if not p.requires_grad and p.grad is not None]
        if frozen_bad:
            raise AssertionError("Frozen weights received gradients.")
        if step in (0, steps-1):
            group_norms = {}
            for group, prefix in (("backbone_or_adaptation", "lm."), ("task_components", None)):
                square_sum = sum(float(p.grad.detach().float().square().sum())
                    for n, p in model.named_parameters() if p.requires_grad and p.grad is not None
                    and (n.startswith(prefix) if prefix else not n.startswith("lm.")))
                group_norms[group] = math.sqrt(square_sum)
            if not all(v > 0 and math.isfinite(v) for v in group_norms.values()):
                raise AssertionError("A required learning component has zero or nonfinite gradients.")
            gradient_audit[str(step+1)] = group_norms
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.)
        scaler.step(optimizer)
        scaler.update()
        rows.append({"step": step+1, "loss": float(loss.detach())})
    after = evaluation_loss()
    model.zero_grad(set_to_none=True)
    del optimizer, scaler
    for module, attribute, probability in dropout_backup:
        setattr(module, attribute, probability)
    return {"status": "passed" if after < before-1e-5 else "needs_review",
            "before_gold_ce": before, "after_gold_ce": after, "steps": steps,
            "seconds": time.monotonic()-started, "curve": rows,
            "gradient_group_norms": gradient_audit,
            "scope": "two training cases, short chunks; wiring/memorization only",
            "temporary_dropout_disabled_and_restored": True,
            "student_teacher_targets": "synthetic_wiring_only" if model.architecture == "efficient" else None}


def checkpoint_roundtrip(model, temporary_dir):
    path = Path(temporary_dir) / "temporary_trainable_roundtrip.pt"
    state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    torch.save({"trainable": state, "architecture": model.architecture}, path)
    expected = trainable_hash(model)
    with torch.no_grad():
        next(p for p in model.parameters() if p.requires_grad).add_(1)
        saved = torch.load(path, map_location="cpu", weights_only=True)
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(saved["trainable"][name])
    if trainable_hash(model) != expected:
        raise AssertionError("Trainable state checkpoint round trip failed.")
    path.unlink()
    return {"status": "passed", "checkpoint_scope": "trainable weights plus original Phase 1 dependency",
            "optimizer_resume_check": "not performed; full trainer is not part of preparation"}


def resource_probes(model, train_documents, settings, device):
    """Stress maximum chunks; successful probes include allocated AdamW state."""
    candidates = settings["resource_candidates"]
    results = []
    parallel = nn.DataParallel(model, device_ids=[0, 1]) if device.type == "cuda" and torch.cuda.device_count() >= 2 else model
    model.train()
    model.use_amp = device.type == "cuda"
    overall_started = time.monotonic()
    for candidate in candidates:
        if time.monotonic()-overall_started > settings.get("resource_budget_minutes", 12)*60:
            results.append(dict(candidate, status="skipped_budget", reason="Per-architecture resource budget reached."))
            continue
        print("Resource probe:", model.architecture, candidate, flush=True)
        optimizer = scaler = batch = labels = output = loss = None
        try:
            width, count, micro = candidate["total_tokens"], candidate["max_chunks"], candidate["chunk_microbatch"]
            model.chunk_microbatch = micro
            # Controlled full-length stress cases; no labels or tokens from development/test.
            source = train_documents[0]["token_ids"]
            required = (width-2)*count
            repeated = np.resize(source, required).astype(np.int32)
            stress = [{"token_ids": repeated, "gold": i, "id": "stress_"+str(i)} for i in (0, 1)]
            batch, labels = collate_cases(stress, total_tokens=width, overlap=0, max_chunks=count)
            batch, labels = batch_to(batch, device), labels.to(device)
            if device.type == "cuda":
                for i in range(min(2, torch.cuda.device_count())):
                    torch.cuda.synchronize(i)
                    torch.cuda.reset_peak_memory_stats(i)
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.0)
            scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
            started = time.monotonic()
            samples = []
            for step in range(2):
                optimizer.zero_grad(set_to_none=True)
                one = time.monotonic()
                output = parallel(**batch)
                loss = F.cross_entropy(output["logits"], labels)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite resource-probe loss.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                if any(p.grad is not None and not bool(torch.isfinite(p.grad).all()) for p in model.parameters()):
                    raise ValueError("Nonfinite resource-probe gradients.")
                scaler.step(optimizer)
                scaler.update()
                if device.type == "cuda":
                    torch.cuda.synchronize(0)
                    if torch.cuda.device_count() > 1:
                        torch.cuda.synchronize(1)
                samples.append(time.monotonic()-one)
            devices = sorted(set(output["device_ids"].detach().cpu().tolist()))
            memory = [{"device": i, "peak_allocated_gib": torch.cuda.max_memory_allocated(i)/2**30,
                       "peak_reserved_gib": torch.cuda.max_memory_reserved(i)/2**30,
                       "total_gib": torch.cuda.get_device_properties(i).total_memory/2**30}
                      for i in range(torch.cuda.device_count())] if device.type == "cuda" else []
            results.append(dict(candidate, status="passed", seconds_per_two_cases=samples[-1],
                cold_step_seconds=samples[0], devices_observed=devices, memory=memory,
                allocated_adamw_state=True, zero_lr_preserves_parameters=True,
                scope="maximum chunks and length, one case per GPU, two forward/backward/AdamW steps"))
        except torch.cuda.OutOfMemoryError:
            results.append(dict(candidate, status="oom", reason="CUDA out of memory; smaller candidates still tested."))
        except Exception as exc:
            results.append(dict(candidate, status="failed", reason=safe_error(exc)))
        finally:
            model.zero_grad(set_to_none=True)
            del optimizer, scaler, batch, labels, output, loss
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
    del parallel
    model.chunk_microbatch = settings["chunk_microbatch"]
    return results


def architecture_probe(architecture, checkpoint_path, signature, settings, documents, output,
                       device=torch.device("cuda:0")):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    model = lm = None
    report = {"architecture": architecture, "status": "running"}
    try:
        seed_all(settings["seed"])
        lm, identity = load_base(checkpoint_path, architecture, signature)
        report["phase1"] = identity
        if architecture == "efficient":
            old_parameters = identity["original_parameters"]
            original = lm
            lm = compact_efficient(original, settings["student_layers"])
            # All copied values are compared, including every retained block parameter.
            for new, old in enumerate(settings["student_layers"]):
                for name, tensor in lm.blocks[new].state_dict().items():
                    if not torch.equal(tensor, original.blocks[old].state_dict()[name]):
                        raise AssertionError("Student block inheritance mismatch.")
            if not torch.equal(lm.tok_emb.weight, original.tok_emb.weight):
                raise AssertionError("Student embedding inheritance mismatch.")
            report["student"] = {"retained_zero_based_layers": settings["student_layers"],
                "original_backbone_parameters": old_parameters,
                "student_backbone_parameters": sum(p.numel() for p in lm.parameters()),
                "copied_block_and_embedding_values_verified": True}
            del original
            gc.collect()
        report["native_readout"] = native_readout_check(lm, architecture, device)
        # Compare initialized PEFT paths against the unadapted readout before learning.
        example_ids = torch.tensor([[2, 9, 12, 7, 3]], device=device)
        example_lengths = torch.tensor([5], device=device)
        with torch.no_grad():
            before = chunk_states(lm, architecture, example_ids, example_lengths)
        model = CaseClassifier(lm, architecture, settings).to(device)
        lm = None
        model.eval()
        with torch.no_grad():
            after = chunk_states(model.lm, architecture, example_ids, example_lengths)
        for a, b in zip(before, after):
            torch.testing.assert_close(a, b, rtol=0, atol=0)
        report["adaptation_initial_identity"] = "passed"
        report["parameters"] = trainability_audit(model)
        report["initial_trainable_sha256"] = trainable_hash(model)
        train_docs = [d for d in documents if d["split"] == "train"]
        tiny_docs = [min((d for d in train_docs if d["gold"] == label),
                         key=lambda d: len(d["token_ids"])) for label in (0, 1)]
        # Different valid lengths/chunk counts exercise masking.
        tiny_docs = [dict(d, token_ids=d["token_ids"][:(35 if i == 0 else 93)]) for i, d in enumerate(tiny_docs)]
        batch, labels = collate_cases(tiny_docs, total_tokens=48, overlap=4, max_chunks=2)
        batch, labels = batch_to(batch, device), labels.to(device)
        report["padding_and_pooling"] = padding_check(model, batch)
        report["two_gpu"] = two_gpu_check(model, batch, labels)
        # Temporary optimizer updates are isolated to this disposable model instance.
        report["temporary_learning"] = temporary_learning(model, batch, labels, settings["smoke_steps"])
        report["checkpoint_roundtrip"] = checkpoint_roundtrip(model, output)
        del batch, labels, before, after
        if settings["run_resource_probes"]:
            report["resources"] = resource_probes(model, train_docs, settings, device)
        else:
            report["resources"] = []
        passed = [r for r in report["resources"] if r["status"] == "passed"]
        good_margin = [r for r in passed if not r["memory"] or
                       max(m["peak_allocated_gib"]/m["total_gib"] for m in r["memory"]) < .85]
        choice = max(good_margin or passed, key=lambda r: (r["total_tokens"], r["max_chunks"])) if passed else None
        report["recommended_resource_configuration"] = choice
        blockers = []
        if report["temporary_learning"]["status"] != "passed":
            blockers.append("Tiny training-set loss did not improve; inspect the curve.")
        if settings["require_two_gpus"] and report["two_gpu"]["status"] != "passed":
            blockers.append("Required two-GPU gradient check was not completed.")
        if settings["run_resource_probes"] and not passed:
            blockers.append("No resource configuration passed.")
        if choice:
            count = len(train_docs)
            report["runtime_projection"] = {
                "stress_based_train_epoch_minutes": choice["seconds_per_two_cases"]*math.ceil(count/2)/60,
                "assumptions": "One case/GPU; padded maximum-chunk stress workload. Excludes validation, exports and I/O.",
                "not_a_measured_full_epoch": True}
        report["training_blockers"] = blockers
        report["status"] = "passed" if not blockers else "needs_review"
        report["temporary_models_discarded"] = True
        report["real_teacher_cache_created"] = False
    except KeyboardInterrupt:
        report.update(status="interrupted", error="Stopped by user; partial reports retained.")
        raise
    except Exception as exc:
        report.update(status="failed", error_type=type(exc).__name__, error=safe_error(exc))
    finally:
        if model is not None:
            model.zero_grad(set_to_none=True)
        del model, lm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        write_json(output / "probe_report.json", report)
    return report


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_fastcache.py
"""Architecture-independent packed tokens: a few sequential files instead of 40k NPZ reads."""
import json
import shutil
import time
import zipfile
from pathlib import Path
import numpy as np
from cjpe_data import read_json, write_json, sha256

MARKER = "cjpe_fast_cache.json"
FILES = ["token_cache/index.json", "reports/tokenizer.json",
         "official_data/Phase_4_CJPE/download_manifest.json",
         "fast_cache/token_ids.npy", "fast_cache/offsets.npy"]


def export_fast_cache(root, archive):
    root, archive = Path(root), Path(archive)
    manifest = read_json(root/MARKER)
    if archive.exists():
        try:
            with zipfile.ZipFile(archive) as z:
                existing = json.loads(z.read("phase4_cjpe_fast_cache/"+MARKER))
            if existing == manifest:
                print("Reusable fast-cache ZIP:", archive, flush=True)
                return archive
        except (OSError, ValueError, KeyError, zipfile.BadZipFile):
            pass
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.with_name(archive.name+".tmp")
    print("Creating reusable fast-cache ZIP:", archive,
          "| one-time compression of the token array", flush=True)
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for name in FILES+[MARKER]:
            print("Packing:", name, flush=True)
            z.write(root/name, "phase4_cjpe_fast_cache/"+name)
    temporary.replace(archive)
    print("Fast-cache ZIP ready:", archive, "|", round(archive.stat().st_size/2**20), "MiB", flush=True)
    return archive


def stage_fast_cache(source, destination):
    """Keep memory-mapped tokens on working storage, avoiding mounted-input random reads."""
    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        return source
    wanted = read_json(source/MARKER)
    if (destination/MARKER).exists():
        if read_json(destination/MARKER) != wanted:
            raise ValueError("A different fast cache occupies the working cache folder. Use another WORK_ROOT.")
        return destination
    destination.mkdir(parents=True, exist_ok=True)
    print("Staging the packed data on working storage:", destination, flush=True)
    for name in FILES:
        original, target = source/name, destination/name
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name+".tmp")
        print("Copying:", name, "|", round(original.stat().st_size/2**20, 1), "MiB", flush=True)
        copied, last = 0, time.monotonic()
        with open(original, "rb") as src, open(temporary, "wb") as out:
            while True:
                block = src.read(8*2**20)
                if not block:
                    break
                out.write(block)
                copied += len(block)
                if time.monotonic()-last >= 20:
                    print("Copied", round(copied/2**20), "MiB of", name, flush=True)
                    last = time.monotonic()
        temporary.replace(target)
    write_json(destination/MARKER, wanted)
    return destination


def load_fast_cache(root):
    root = Path(root)
    manifest = read_json(root/MARKER)
    if manifest.get("format_version") != 1:
        raise ValueError("Unsupported packed-cache version.")
    print("Loading packed CJPE data; verifying", len(FILES), "sequential files.", flush=True)
    for name in FILES:
        print("Checking packed file:", name, flush=True)
        if sha256(root/name) != manifest["sha256"][name]:
            raise ValueError("Packed data changed: "+name)
    index = read_json(root/"token_cache/index.json")
    official = read_json(root/"official_data/Phase_4_CJPE/download_manifest.json")
    tokenizer = read_json(root/"reports/tokenizer.json")
    if (index.get("record_identity_version") != 2 or
            index["fingerprint"] != manifest["cache_fingerprint"] or
            official["sha256"] != index["identity"]["official_data_sha256"] or
            tokenizer["sha256"] != index["identity"]["tokenizer_sha256"]):
        raise ValueError("Packed data/tokenizer identities disagree.")
    tokens = np.load(root/"fast_cache/token_ids.npy", mmap_mode="r", allow_pickle=False)
    offsets = np.load(root/"fast_cache/offsets.npy", allow_pickle=False)
    counts = np.asarray([r["content_tokens"] for r in index["cases"]], dtype=np.int64)
    if (tokens.ndim != 1 or tokens.dtype != np.int32 or offsets.dtype != np.int64 or
            offsets.shape != (len(counts)+1,) or offsets[0] != 0 or offsets[-1] != len(tokens) or
            not np.array_equal(np.diff(offsets), counts)):
        raise ValueError("Invalid packed token layout.")
    documents, seen = [], set()
    for i, row in enumerate(index["cases"]):
        if row["row_uid"] in seen:
            raise ValueError("Duplicate packed row_uid.")
        seen.add(row["row_uid"])
        documents.append(dict(row, token_ids=tokens[int(offsets[i]):int(offsets[i+1])]))
    identity = {"data": official, "tokenizer_sha256": index["identity"]["tokenizer_sha256"],
                "cache_fingerprint": index["fingerprint"]}
    print("Packed data ready:", len(documents), "cases; token IDs, labels and row order preserved.", flush=True)
    return documents, identity


def build_fast_cache(source, documents, identity, destination, archive=None):
    """Call only after the original loader has verified every source cache file."""
    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        raise ValueError("Packed output must be separate from the original prepared folder.")
    if (destination/MARKER).exists():
        saved = read_json(destination/MARKER)
        if saved["cache_fingerprint"] != identity["cache_fingerprint"]:
            raise ValueError("Another dataset occupies the working fast-cache folder.")
        load_fast_cache(destination)
        if archive is not None:
            export_fast_cache(destination, archive)
        return destination
    destination.mkdir(parents=True, exist_ok=True)
    print("Building the reusable packed cache once; no retokenization or model training.", flush=True)
    for name in FILES[:3]:
        target = destination/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source/name, target)
    folder = destination/"fast_cache"
    folder.mkdir(exist_ok=True)
    offsets = np.zeros(len(documents)+1, dtype=np.int64)
    offsets[1:] = np.cumsum([len(d["token_ids"]) for d in documents], dtype=np.int64)
    np.save(folder/"offsets.npy", offsets, allow_pickle=False)
    token_file = folder/"token_ids.npy"
    packed = np.lib.format.open_memmap(token_file, mode="w+", dtype=np.int32, shape=(int(offsets[-1]),))
    last = time.monotonic()
    for i, row in enumerate(documents):
        values = np.asarray(row["token_ids"])
        if values.ndim != 1 or values.dtype != np.int32:
            raise ValueError("The prepared tokens must be the original one-dimensional int32 IDs.")
        packed[int(offsets[i]):int(offsets[i+1])] = values
        if (i+1) % 5000 == 0 or time.monotonic()-last >= 20:
            print("Packed cases:", i+1, "/", len(documents), flush=True)
            last = time.monotonic()
    packed.flush()
    del packed
    hashes = {}
    for name in FILES:
        print("Recording packed file checksum:", name, flush=True)
        hashes[name] = sha256(destination/name)
    write_json(destination/MARKER, {"format_version": 1, "cache_fingerprint": identity["cache_fingerprint"],
        "source_index_sha256": hashes["token_cache/index.json"], "case_rows": len(documents),
        "content_tokens": int(offsets[-1]), "sha256": hashes,
        "purpose": "verified_original_cjpe_tokens_for_all_architectures"})
    if archive is not None:
        export_fast_cache(destination, archive)
    return destination


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_reuse.py
"""Load the prepared training inputs in a later notebook without retokenizing."""
from pathlib import Path
import hashlib
import io
import time
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import numpy as np
from cjpe_data import read_json, sha256
from cjpe_fastcache import (MARKER, build_fast_cache, export_fast_cache, load_fast_cache,
                            stage_fast_cache)


def load_prepared_cases(prepared_root, fast_cache_dir=None, fast_cache_zip=None, num_workers=16):
    root = Path(prepared_root)
    started = last_progress = time.monotonic()
    print("Prepared input:", root, "| reading cache index", flush=True)
    if not (root/"token_cache"/"index.json").exists():
        choices = list(root.rglob("token_cache/index.json"))
        if len(choices) != 1:
            raise ValueError("Choose the extracted phase4_cjpe_prepared folder.")
        root = choices[0].parent.parent
    if (root/MARKER).exists():
        packed_root = stage_fast_cache(root, fast_cache_dir) if fast_cache_dir is not None else root
        documents, identity = load_fast_cache(packed_root)
        if fast_cache_zip is not None:
            export_fast_cache(packed_root, fast_cache_zip)
        return documents, identity
    cache = root/"token_cache"
    index = read_json(cache/"index.json")
    if index.get("record_identity_version") != 2:
        raise ValueError("This older cache lacks unique row identities. Run the corrected preparation notebook.")
    if fast_cache_dir is not None and (Path(fast_cache_dir)/MARKER).exists():
        saved = read_json(Path(fast_cache_dir)/MARKER)
        if saved["source_index_sha256"] != sha256(cache/"index.json"):
            raise ValueError("The working packed cache was built from another preparation index.")
        documents, identity = load_fast_cache(fast_cache_dir)
        if fast_cache_zip is not None:
            export_fast_cache(fast_cache_dir, fast_cache_zip)
        return documents, identity
    data_root = root/"official_data"/"Phase_4_CJPE"
    identity = read_json(data_root/"download_manifest.json")
    if identity["sha256"] != index["identity"]["official_data_sha256"]:
        raise ValueError("Prepared data and cache identities disagree.")
    for suffix, expected in identity["sha256"].items():
        print("Verifying official data:", suffix, flush=True)
        if sha256(data_root/suffix) != expected:
            raise ValueError("Prepared official JSON changed: " + suffix)
    rows = index["cases"]
    if len({row["row_uid"] for row in rows}) != len(rows):
        raise ValueError("Duplicate prepared case identity.")
    total = len(index["cases"])
    workers = max(1, int(num_workers))
    print("Loading and verifying", total, "cached cases with", workers,
          "CPU I/O workers; each file is read once.", flush=True)

    def read_case(row):
        path = cache/row["token_file"]
        blob = path.read_bytes()
        if hashlib.sha256(blob).hexdigest() != index["cache_sha256"][row["token_file"]]:
            raise ValueError("Prepared token-cache bytes changed.")
        with np.load(io.BytesIO(blob), allow_pickle=False) as z:
            ids = z["ids"].copy()
        if ids.ndim != 1 or len(ids) != row["content_tokens"]:
            raise ValueError("Prepared token-cache length mismatch.")
        return dict(row, token_ids=ids)

    documents = [None]*total
    pool = ThreadPoolExecutor(max_workers=workers)
    pending, next_row, completed, last_count = {}, 0, 0, 0
    try:
        while next_row < min(total, workers*2):
            pending[pool.submit(read_case, rows[next_row])] = next_row
            next_row += 1
        while pending:
            finished, _ = wait(pending, timeout=5, return_when=FIRST_COMPLETED)
            for future in finished:
                position = pending.pop(future)
                documents[position] = future.result()
                completed += 1
                if next_row < total:
                    pending[pool.submit(read_case, rows[next_row])] = next_row
                    next_row += 1
            now = time.monotonic()
            if completed-last_count >= 1000 or completed == total or now-last_progress >= 20:
                print("Cached cases:", completed, "/", total,
                      "| elapsed", round((now-started)/60, 1), "min", flush=True)
                last_progress, last_count = now, completed
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    result_identity = {"data": identity, "tokenizer_sha256": index["identity"]["tokenizer_sha256"],
                       "cache_fingerprint": index["fingerprint"]}
    if fast_cache_dir is not None:
        build_fast_cache(root, documents, result_identity, fast_cache_dir, fast_cache_zip)
    return documents, result_identity


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/evaluate_phase4.py
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_train_io.py
"""Shared training inputs, duplicate policy, initialization and checkpoint helpers."""
import copy
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
from cjpe_data import read_json, write_json, sha256, discover
from cjpe_reuse import load_prepared_cases
from cjpe_checks import load_base, seed_all
from cjpe_models import CaseClassifier, compact_efficient, trainability_audit


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def extract_own_zip(path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    print("Opening archive:", path, "| destination:", destination, flush=True)
    started = last_progress = time.monotonic()
    with zipfile.ZipFile(path) as z:
        entries = z.infolist()
        for number, info in enumerate(entries, 1):
            now = time.monotonic()
            if number == 1 or number % 2000 == 0 or now-last_progress >= 20:
                print("Archive entries:", number, "/", len(entries),
                      "| elapsed", round((now-started)/60, 1), "min", flush=True)
                last_progress = now
            if info.is_dir():
                continue
            target = (destination/info.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe ZIP member.")
            if target.exists() and target.stat().st_size == info.file_size:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name+".tmp")
            if info.file_size >= 256*2**20:
                print("Extracting", info.filename, "|", round(info.file_size/2**20), "MiB", flush=True)
            with z.open(info) as source, open(temporary, "wb") as out:
                shutil.copyfileobj(source, out, 8*2**20)
            temporary.replace(target)
    return destination


def prepared_root(config):
    supplied = config["prepared_input"]
    if supplied:
        p = Path(supplied)
        if p.is_file():
            p = extract_own_zip(p, Path(config["work_root"])/"extracted_preparation")
        matches = [p/"token_cache"/"index.json"] if (p/"token_cache"/"index.json").exists() else list(p.rglob("token_cache/index.json"))
        if len(matches) != 1:
            raise ValueError("PREPARED_INPUT needs one prepared token_cache/index.json.")
        return matches[0].parent.parent
    packed_working = Path(config["work_root"])/"prepared_fast"
    if (packed_working/"cjpe_fast_cache.json").exists():
        print("Reusing the local packed dataset:", packed_working, flush=True)
        return packed_working
    locations = [Path("/kaggle/working/cs781_phase4_prep"),
                 Path(config["work_root"])/"extracted_preparation"]
    for folder in locations:
        if folder.exists():
            print("Searching for prepared cache in:", folder, flush=True)
            matches = sorted(folder.rglob("token_cache/index.json"),
                key=lambda p: ((p.parent.parent/"cjpe_fast_cache.json").exists(), p.stat().st_mtime_ns), reverse=True)
            if matches:
                return matches[0].parent.parent
    print("Searching attached inputs; skipping the per-case cache directories during discovery.", flush=True)
    packed_roots, original_roots, packed_archives, original_archives = [], [], [], []
    for folder_name, folders, files in os.walk("/kaggle/input"):
        folder = Path(folder_name)
        if "token_cache" in folders and (folder/"token_cache"/"index.json").exists():
            (packed_roots if "cjpe_fast_cache.json" in files else original_roots).append(folder)
        if "phase4_cjpe_fast_cache.zip" in files:
            packed_archives.append(folder/"phase4_cjpe_fast_cache.zip")
        if "phase4_cjpe_prepared_data.zip" in files:
            original_archives.append(folder/"phase4_cjpe_prepared_data.zip")
        folders[:] = [name for name in folders if name not in ("token_cache", "fast_cache", "__pycache__")]
    if packed_roots:
        return sorted(packed_roots)[0]
    archives = packed_archives
    if not archives:
        archives = original_archives
    if not archives and original_roots:
        return sorted(original_roots)[0]
    if len(archives) == 1:
        target = extract_own_zip(archives[0], Path(config["work_root"])/"extracted_preparation")
        matches = list(target.rglob("token_cache/index.json"))
        if len(matches) == 1:
            return matches[0].parent.parent
    raise FileNotFoundError("Attach phase4_cjpe_fast_cache.zip or phase4_cjpe_prepared_data.zip, or set PREPARED_INPUT to its extracted folder. "
                            "The existing preparation working folder is also supported.")


def identical_key(document, include_gold=False):
    key = (document["id"], document["text_sha256"])
    return key+(document["gold"],) if include_gold else key


def select_train(documents, deduplicate):
    rows = [d for d in documents if d["split"] == "train"]
    if not deduplicate:
        return rows
    seen, selected = set(), []
    for row in rows:
        key = identical_key(row, include_gold=True)
        if key not in seen:
            seen.add(key)
            selected.append(row)
    return selected


def prediction_groups(documents):
    """Compute identical inputs once; expand results back to every original row."""
    unique, lookup, inverse = [], {}, []
    for row in documents:
        key = identical_key(row)
        if key not in lookup:
            lookup[key] = len(unique)
            unique.append(row)
        inverse.append(lookup[key])
    return unique, np.asarray(inverse, dtype=np.int64)


def class_weights(rows, mode):
    counts = np.bincount([d["gold"] for d in rows], minlength=2)
    if not np.all(counts):
        raise ValueError("Both training labels must be present.")
    if mode is None or mode == "none":
        weights = None
    elif mode == "balanced":
        weights = len(rows)/(2*counts)
    elif mode == "sqrt":
        weights = np.sqrt(counts.max()/counts)
        weights /= weights.mean()
    else:
        raise ValueError("CLASS_WEIGHTS must be None, 'balanced' or 'sqrt'.")
    return counts.tolist(), None if weights is None else torch.tensor(weights, dtype=torch.float32)


def context(config, source_dir):
    source_dir = Path(source_dir)
    reference = read_json(source_dir/"cjpe_reference.json")
    print("Locating prepared CJPE inputs...", flush=True)
    root = prepared_root(config)
    rows, identity = load_prepared_cases(root, Path(config["work_root"])/"prepared_fast",
        Path(config["output_root"])/"phase4_cjpe_fast_cache.zip", config.get("data_read_workers", 16))
    if identity["cache_fingerprint"] != reference["prepared_cache_fingerprint"]:
        raise ValueError("Prepared data differ from the uploaded passing preparation. Use the matching prepared-data ZIP.")
    report = read_json(root/"reports"/"tokenizer.json")
    if report["sha256"] != identity["tokenizer_sha256"]:
        raise ValueError("Saved tokenizer identity and token cache disagree.")
    architecture = config["architecture"]
    spec = reference["phase1_inputs"][architecture]
    print("Locating original Phase 1 checkpoint for", architecture, flush=True)
    checkpoint = discover("best_model.pt", "/kaggle/input", config["phase1_input"],
        [spec["hint"], spec["dataset"]], Path(config["work_root"])/"extracted_phase1")
    print("Verifying original checkpoint:", checkpoint, flush=True)
    if sha256(checkpoint) != spec["sha256"]:
        raise ValueError("Select the original Phase 1 checkpoint used by the passed preparation.")
    train = select_train(rows, config["deduplicate_identical_training_rows"])
    dev, test = [[d for d in rows if d["split"] == split] for split in ("dev", "test")]
    if len({d["id"] for d in test}) != len(test):
        raise ValueError("Test submission IDs must be unique.")
    counts, weights = class_weights(train, config["class_weights"])
    rng = np.random.default_rng(config["seed"]+901)
    requested = config["train_eval_cases"]
    n = len(train) if requested is None else min(requested, len(train))
    chosen = sorted(rng.choice(len(train), size=n, replace=False).tolist()) if n else []
    selected_fingerprint = digest([d["row_uid"] for d in train])
    info = {"prepared_root": str(root), "data_identity": identity, "phase1_path": str(checkpoint),
        "phase1_sha256": spec["sha256"], "architecture": architecture,
        "all_train_rows": sum(d["split"] == "train" for d in rows),
        "effective_train_rows": len(train), "dev_rows": len(dev), "test_rows": len(test),
        "unique_dev_forward_passes": len(prediction_groups(dev)[0]),
        "deduplicate_identical_training_rows": config["deduplicate_identical_training_rows"],
        "train_selection_fingerprint": selected_fingerprint, "train_class_counts": counts,
        "class_weights": weights.tolist() if weights is not None else None,
        "tokenizer_signature": report["signature"], "source_sha256": {p.name: sha256(p) for p in source_dir.glob("*.py")}}
    return {"rows": rows, "train": train, "dev": dev, "test": test,
        "train_eval": [train[i] for i in chosen], "class_weights": weights,
        "signature": report["signature"], "checkpoint": checkpoint, "info": info}


def build_classifier(ctx, config, model_settings=None):
    seed_all(config["seed"])
    lm, phase1 = load_base(ctx["checkpoint"], config["architecture"], ctx["signature"])
    settings = copy.deepcopy(model_settings or config["model_settings"])
    if config["architecture"] == "efficient":
        original = lm
        lm = compact_efficient(original, settings["student_layers"])
        del original
    model = CaseClassifier(lm, config["architecture"], settings)
    model.use_amp = config["amp"]
    gc.collect()
    return model, {"phase1": phase1, "parameters": trainability_audit(model), "model_settings": settings}


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def set_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available():
        for i, value in enumerate(state.get("cuda", [])[:torch.cuda.device_count()]):
            torch.cuda.set_rng_state(value.cpu(), i)


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(model, state):
    parameters = {n: p for n, p in model.named_parameters() if p.requires_grad}
    if set(parameters) != set(state):
        raise ValueError("Trainable tensor set changed. Use a new RUN_TAG for a different adaptation scope/model shape.")
    with torch.no_grad():
        for name, parameter in parameters.items():
            if parameter.shape != state[name].shape:
                raise ValueError("Model shape changed at "+name+". Use a new RUN_TAG.")
            parameter.copy_(state[name])


def atomic_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def resume_files(config, run_dir):
    run_dir = Path(run_dir)
    supplied = config["resume_input"]
    have_local = ((run_dir/"last.pt").exists() or
                  (config["action"] == "export_only" and (run_dir/"best.pt").exists()))
    if supplied and not (have_local and config["auto_resume"]):
        p = Path(supplied)
        if p.suffix.lower() == ".zip":
            p = extract_own_zip(p, Path(config["work_root"])/"restored_backup"/digest(str(p))[:10])
        if p.is_file():
            if p.name not in ("last.pt", "best.pt"):
                raise ValueError("RESUME_INPUT must be this run's backup, run folder, last.pt or best.pt.")
            source = p.parent
        else:
            candidates = list(p.rglob("last.pt")) or list(p.rglob("best.pt"))
            if len(candidates) != 1:
                raise ValueError("Choose one exact run backup/folder.")
            source = candidates[0].parent
        for name in ("last.pt", "best.pt"):
            if (source/name).exists() and (source/name).resolve() != (run_dir/name).resolve():
                shutil.copy2(source/name, run_dir/name)
    if config["action"] == "export_only":
        if not (run_dir/"best.pt").exists():
            raise FileNotFoundError("Export-only needs this run's best.pt or backup.")
    elif supplied and not (run_dir/"last.pt").exists():
        raise ValueError("Training resume needs last.pt (optimizer/progress state). Use ACTION='export_only' for best.pt alone.")
    elif (run_dir/"last.pt").exists() and not config["auto_resume"]:
        raise ValueError("This RUN_TAG already has a checkpoint. Enable AUTO_RESUME or use a new RUN_TAG.")
    return run_dir/"last.pt" if (run_dir/"last.pt").exists() else None


def read_teacher_cache(path, all_rows, identity):
    p = Path(path)
    if p.is_dir():
        choices = list(p.rglob("teacher_logits_train.npz"))
        if len(choices) != 1:
            raise ValueError("Choose one genuine teacher_logits_train.npz cache.")
        p = choices[0]
    if p.suffix.lower() == ".zip":
        raise ValueError("Extract the teacher-cache ZIP or attach its output folder.")
    with np.load(p, allow_pickle=False) as z:
        meta = json.loads(str(z["metadata"]))
        uids, logits, labels = z["row_uids"].tolist(), z["logits"].copy(), z["labels"].tolist()
    if (meta.get("purpose") != "trained_cjpe_teacher" or meta.get("split") != "train" or
            meta.get("trained_updates", 0) < 1 or meta.get("class_order") != [0, 1]):
        raise ValueError("KD needs a genuine trained CJPE teacher cache, not synthetic preflight targets.")
    if meta["cache_fingerprint"] != identity["cache_fingerprint"]:
        raise ValueError("Teacher and student data/tokenizer identities differ.")
    rows = [d for d in all_rows if d["split"] == "train"]
    expected = {d["row_uid"]: d["gold"] for d in rows}
    if len(uids) != len(set(uids)) or set(uids) != set(expected) or logits.shape != (len(rows), 2):
        raise ValueError("Teacher cache must cover all original training row_uids exactly once.")
    if not np.isfinite(logits).all() or any(expected[u] != y for u, y in zip(uids, labels)):
        raise ValueError("Teacher cache labels/logits are invalid or misaligned.")
    return {u: logits[i] for i, u in enumerate(uids)}, dict(meta, cache_sha256=sha256(p))


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_trainer.py
"""One case-classification trainer for decoder, adapters, LoRMA+ and a KD student."""
import copy
import contextlib
import gc
import json
import math
import random
import time
import zipfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, precision_recall_fscore_support, roc_auc_score)
from cjpe_data import collate_cases, write_json, read_json, sha256, evaluation_id, safe_error
from cjpe_train_io import (prediction_groups, rng_state, set_rng, trainable_state,
                           load_trainable, atomic_torch, digest)
from cjpe_checks import trainable_hash


class TrainableEMA:
    """Average only updated parameters; frozen Phase-1 weights are shared unchanged."""
    def __init__(self, model, decay):
        if not 0 <= decay < 1:
            raise ValueError("EMA_DECAY must be in [0, 1).")
        self.decay, self.updates = float(decay), 0
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow:
                if self.updates == 0:
                    self.shadow[n].copy_(p)
                else:
                    self.shadow[n].lerp_(p.detach(), 1-self.decay)
        self.updates += 1

    def state_dict(self):
        return {"decay": self.decay, "updates": self.updates,
                "shadow": {n: p.detach().cpu().clone() for n, p in self.shadow.items()}}

    @torch.no_grad()
    def load_state_dict(self, saved):
        if set(saved["shadow"]) != set(self.shadow):
            raise ValueError("EMA parameter names changed. Use a new RUN_TAG.")
        self.updates = int(saved["updates"])
        for n, p in self.shadow.items():
            if p.shape != saved["shadow"][n].shape:
                raise ValueError("EMA parameter shapes changed. Use a new RUN_TAG.")
            p.copy_(saved["shadow"][n])

    @contextlib.contextmanager
    @torch.no_grad()
    def applied(self, model):
        raw = {n: p.detach().clone() for n, p in model.named_parameters() if n in self.shadow}
        try:
            for n, p in model.named_parameters():
                if n in self.shadow:
                    p.copy_(self.shadow[n])
            yield
        finally:
            for n, p in model.named_parameters():
                if n in raw:
                    p.copy_(raw[n])


def geometry(config):
    return {key: config[key] for key in ("total_tokens", "overlap", "max_chunks", "policy")}


def move(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def runner_for(model):
    if next(model.parameters()).is_cuda and torch.cuda.device_count() >= 2:
        return nn.DataParallel(model, device_ids=[0, 1])
    return model


def synchronize(device):
    if device.type == "cuda":
        for i in range(min(2, torch.cuda.device_count())):
            torch.cuda.synchronize(i)


def make_optimizer(model, config, zero_lr=False):
    groups = []
    for name, prefix, rate in (("backbone", True, config["backbone_lr"]),
                               ("head", False, config["head_lr"])):
        parameters = [p for n, p in model.named_parameters()
                      if p.requires_grad and n.startswith("lm.") == prefix]
        if parameters:
            groups.append({"params": parameters, "name": name, "lr": 0.0 if zero_lr else rate})
    return torch.optim.AdamW(groups, betas=tuple(config["adam_betas"]), eps=config["adam_eps"],
                             weight_decay=config["weight_decay"])


def apply_schedule(optimizer, config, samples_seen, train_count):
    total = max(1, config["epochs"]*train_count)
    warm = config["warmup_fraction"]*total
    progress = min(1., samples_seen/total)
    if warm and samples_seen < warm:
        factor = max(1/total, (samples_seen+1)/warm)
    else:
        fraction = (progress-config["warmup_fraction"])/max(1-config["warmup_fraction"], 1e-9)
        factor = config["min_lr_fraction"]+(1-config["min_lr_fraction"])*.5*(1+math.cos(math.pi*max(0., fraction)))
    for group in optimizer.param_groups:
        base = config["backbone_lr"] if group["name"] == "backbone" else config["head_lr"]
        group.update(lr=base*factor, weight_decay=config["weight_decay"],
                     betas=tuple(config["adam_betas"]), eps=config["adam_eps"])


def probabilities(logits):
    return torch.softmax(torch.as_tensor(logits).float(), -1)[:, 1].numpy()


def metrics(labels, logits, threshold=.5):
    labels = np.asarray(labels, dtype=np.int64)
    probability = probabilities(logits)
    pred = (probability >= threshold).astype(np.int64)
    per_p, per_r, per_f, support = precision_recall_fscore_support(labels, pred, labels=[0, 1], zero_division=0)
    return {"macro_f1": float(f1_score(labels, pred, labels=[0, 1], average="macro", zero_division=0)),
        "precision_1": float(precision_score(labels, pred, zero_division=0)),
        "recall_1": float(recall_score(labels, pred, zero_division=0)),
        "accuracy": float((pred == labels).mean()), "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(labels, probability)) if len(np.unique(labels)) == 2 else None,
        "predicted_positive_fraction": float(pred.mean()),
        "confusion_matrix": confusion_matrix(labels, pred, labels=[0, 1]).tolist(),
        "per_class": {str(i): {"precision": float(per_p[i]), "recall": float(per_r[i]),
            "f1": float(per_f[i]), "support": int(support[i])} for i in (0, 1)},
        "cross_entropy": float(F.cross_entropy(torch.as_tensor(logits).float(), torch.as_tensor(labels)))}


@torch.no_grad()
def predict(model, rows, config, batch_size=None):
    unique, inverse = prediction_groups(rows)
    device = next(model.parameters()).device
    runner = runner_for(model)
    previous = model.training
    model.eval()
    outputs = []
    batch_size = batch_size or config["eval_batch_cases"]
    started = last = time.monotonic()
    for start in range(0, len(unique), batch_size):
        batch, _ = collate_cases(unique[start:start+batch_size], **geometry(config))
        output = runner(**move(batch, device))["logits"].detach().cpu().numpy()
        if not np.isfinite(output).all():
            raise FloatingPointError("Nonfinite evaluation logits.")
        outputs.append(output)
        if time.monotonic()-last > 45:
            print("Evaluation:", min(start+batch_size, len(unique)), "/", len(unique), "unique inputs", flush=True)
            last = time.monotonic()
    result = np.concatenate(outputs)[inverse]
    model.train(previous)
    return result, {"rows": len(rows), "unique_forward_inputs": len(unique),
                    "seconds": time.monotonic()-started}


def startup_probe(model, train_rows, config):
    """Two zero-LR updates at the requested maximum shape; no learning sweep."""
    device = next(model.parameters()).device
    saved_rng, old_mode = rng_state(), model.training
    candidates = [config["train_batch_cases"]]
    if config["auto_reduce_batch_on_startup"]:
        candidates += [v for v in (8, 4, 2, 1) if v < candidates[0]]
    attempts = []
    try:
        for batch_size in candidates:
            optimizer = scaler = result = loss = batch = labels = parallel = None
            try:
                model.train()
                model.use_amp = config["amp"]
                content = config["total_tokens"]-2
                source = train_rows[0]["token_ids"]
                stress = [{"id": "startup_"+str(i), "gold": i%2,
                    "token_ids": np.resize(source, content*config["max_chunks"]).astype(np.int32)}
                    for i in range(batch_size)]
                batch, labels = collate_cases(stress, total_tokens=config["total_tokens"], overlap=0,
                    max_chunks=config["max_chunks"], policy="uniform")
                batch, labels = move(batch, device), labels.to(device)
                optimizer = make_optimizer(model, config, zero_lr=True)
                scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and config["amp"])
                parallel = runner_for(model)
                if device.type == "cuda":
                    for i in range(min(2, torch.cuda.device_count())):
                        torch.cuda.reset_peak_memory_stats(i)
                elapsed = []
                for _ in range(2):
                    optimizer.zero_grad(set_to_none=True)
                    synchronize(device)
                    started = time.monotonic()
                    result = parallel(**batch)
                    loss = F.cross_entropy(result["logits"], labels)
                    if not bool(torch.isfinite(loss)):
                        raise FloatingPointError("Nonfinite startup loss.")
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    if any(p.grad is not None and not bool(torch.isfinite(p.grad).all()) for p in model.parameters()):
                        raise FloatingPointError("Nonfinite startup gradient.")
                    scaler.step(optimizer)
                    scaler.update()
                    synchronize(device)
                    elapsed.append(time.monotonic()-started)
                devices = sorted(set(result["device_ids"].detach().cpu().tolist()))
                if config["require_two_gpus"] and devices != [0, 1]:
                    raise ValueError("Startup did not use both GPUs. Set TRAIN_BATCH_CASES to at least 2.")
                row = {"status": "passed", "train_batch_cases": batch_size,
                    "seconds_per_microbatch": elapsed[-1], "devices_observed": devices,
                    "total_tokens": config["total_tokens"], "max_chunks": config["max_chunks"],
                    "memory": [{"device": i, "peak_allocated_gib": torch.cuda.max_memory_allocated(i)/2**30}
                               for i in range(min(2, torch.cuda.device_count()))] if device.type == "cuda" else []}
                attempts.append(row)
                return batch_size, {"status": "passed", "attempts": attempts, "updates_discarded": True,
                    "stress_epoch_minutes": elapsed[-1]*math.ceil(len(train_rows)/batch_size)/60,
                    "estimate_excludes_validation_and_exports": True}
            except torch.cuda.OutOfMemoryError:
                attempts.append({"status": "oom", "train_batch_cases": batch_size})
                print("Startup OOM at batch", batch_size, "; trying a smaller case batch.", flush=True)
            finally:
                model.zero_grad(set_to_none=True)
                del optimizer, scaler, result, loss, batch, labels, parallel
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        raise RuntimeError("No requested case batch fitted. Reduce CHUNK_MICROBATCH or document geometry in the settings cell.")
    finally:
        set_rng(saved_rng)
        model.train(old_mode)


def checkpoint_payload(model, optimizer, scaler, state, config, ctx):
    return {"kind": "resume", "format_version": 2, "architecture": config["architecture"],
        "student_mode": config["student_mode"], "phase1_sha256": ctx["info"]["phase1_sha256"],
        "train_selection_fingerprint": ctx["info"]["train_selection_fingerprint"],
        "model": trainable_state(model), "optimizer": optimizer.state_dict(),
        "ema": ctx["_ema"].state_dict() if ctx.get("_ema") is not None else None,
        "scaler": scaler.state_dict(), "rng": rng_state(), "state": copy.deepcopy(state),
        "config": copy.deepcopy(config), "model_settings": copy.deepcopy(model.settings)}


def save_last(model, optimizer, scaler, state, config, ctx, run_dir):
    atomic_torch(Path(run_dir)/"last.pt", checkpoint_payload(model, optimizer, scaler, state, config, ctx))
    write_json(Path(run_dir)/"history.json", state["history"])
    write_json(Path(run_dir)/"progress.json", {k: v for k, v in state.items() if k not in ("history", "order")})


def load_last(model, optimizer, scaler, config, ctx, path):
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if (bool(saved.get("ema")) != bool(config.get("use_ema", False)) or
            saved["config"].get("selection_metric", "macro_f1") != config.get("selection_metric", "macro_f1")):
        raise ValueError("EMA/selection protocol changed. Use a new RUN_TAG, or the saved protocol for resume.")
    if (saved["architecture"] != config["architecture"] or saved["student_mode"] != config["student_mode"] or
            saved["phase1_sha256"] != ctx["info"]["phase1_sha256"] or
            saved["train_selection_fingerprint"] != ctx["info"]["train_selection_fingerprint"]):
        raise ValueError("Resume architecture, original checkpoint or training rows changed. Use a new RUN_TAG.")
    load_trainable(model, saved["model"])
    if ctx.get("_ema") is not None:
        ctx["_ema"].load_state_dict(saved["ema"])
    optimizer.load_state_dict(saved["optimizer"])
    if saved["scaler"]:
        scaler.load_state_dict(saved["scaler"])
    set_rng(saved["rng"])
    state = saved["state"]
    if saved["config"] != config:
        changed = [k for k in config if config.get(k) != saved["config"].get(k)]
        state["history"].append({"kind": "resume_settings", "update": state["updates"], "changed": changed})
        print("Resumed; current settings apply. Changed fields:", ", ".join(changed), flush=True)
    return state


def validation_event(model, ctx, config, state, run_dir, reason):
    ema = ctx.get("_ema")
    with ema.applied(model) if ema is not None else contextlib.nullcontext():
        return _validation_event(model, ctx, config, state, run_dir, reason)


def _validation_event(model, ctx, config, state, run_dir, reason):
    logits, timing = predict(model, ctx["dev"], config)
    result = metrics([d["gold"] for d in ctx["dev"]], logits)
    row = {"kind": "validation", "epoch": state["epoch"]+1, "update": state["updates"],
        "samples_seen": state["samples_seen"], "reason": reason, "dev": result, "timing": timing}
    selection = config.get("selection_metric", "macro_f1")
    score = result[selection]
    if score is None:
        raise ValueError("Selection requires a development set containing both classes.")
    row.update(selection_metric=selection, selection_score=score,
               evaluated_weights="ema" if ctx.get("_ema") is not None else "raw")
    previous = state.get("best_score", state["best_f1"])
    improved = score > previous+config["early_stop_min_delta"]
    tie = abs(score-previous) < 1e-12 and result["cross_entropy"] < state["best_loss"]
    if improved or tie:
        state["best_score"] = score
        state["best_f1"], state["best_loss"] = result["macro_f1"], result["cross_entropy"]
        state["epoch_improved"] |= improved
        selected = {"kind": "selected", "architecture": config["architecture"],
            "student_mode": config["student_mode"], "model": trainable_state(model),
            "model_settings": copy.deepcopy(model.settings), "config": copy.deepcopy(config),
            "phase1_sha256": ctx["info"]["phase1_sha256"], "data_identity": ctx["info"]["data_identity"],
            "trained_updates": state["updates"], "epoch": state["epoch"]+1, "metrics": result,
            "selection_metric": selection, "selection_score": score,
            "evaluated_weights": row["evaluated_weights"]}
        atomic_torch(Path(run_dir)/"best.pt", selected)
        row["new_selected_checkpoint"] = True
    if reason == "epoch_end" and ctx["train_eval"]:
        train_logits, _ = predict(model, ctx["train_eval"], config)
        row["train_eval_subset"] = metrics([d["gold"] for d in ctx["train_eval"]], train_logits)
        row["train_eval_subset_cases"] = len(ctx["train_eval"])
        print("Training evaluation |", len(ctx["train_eval"]), "fixed training cases | macro-F1",
              round(row["train_eval_subset"]["macro_f1"], 5), "| CE",
              round(row["train_eval_subset"]["cross_entropy"], 4),
              "| development macro-F1", round(result["macro_f1"], 5), flush=True)
    state["history"].append(row)
    print("Validation | epoch", row["epoch"], "| update", row["update"],
          "| macro-F1", round(result["macro_f1"], 5), "| CE", round(result["cross_entropy"], 4),
          "| AUC", round(result["roc_auc"], 5) if result["roc_auc"] is not None else None,
          "| positive %", round(100*result["predicted_positive_fraction"], 1),
          "| selected", selection, round(state["best_score"], 5),
          "| weights", row["evaluated_weights"], flush=True)
    return row


def train_group(model, runner, rows, weights, teacher, optimizer, scaler, config, runtime):
    device = next(model.parameters()).device
    optimizer.zero_grad(set_to_none=True)
    runtime["phase"], runtime["group_rng"] = "compute", rng_state()
    labels_all = torch.tensor([d["gold"] for d in rows], dtype=torch.long)
    denominator = float(weights.cpu()[labels_all].sum()) if weights is not None else len(rows)
    ce_sum, kd_sum, total_loss = 0., 0., 0.
    kd_weight = config["kd_weight"] if teacher is not None else 0.
    for start in range(0, len(rows), config["train_batch_cases"]):
        part = rows[start:start+config["train_batch_cases"]]
        batch, labels = collate_cases(part, **geometry(config))
        labels = labels.to(device)
        result = runner(**move(batch, device))
        logits = result["logits"]
        ce = F.cross_entropy(logits.float(), labels, weight=weights, reduction="sum",
                             label_smoothing=config["label_smoothing"])
        kd = ce*0
        if teacher is not None:
            t = config["temperature"]
            target = torch.as_tensor(np.stack([teacher[d["row_uid"]] for d in part]), device=device).float()
            kd = F.kl_div(F.log_softmax(logits.float()/t, -1), torch.softmax(target/t, -1),
                          reduction="sum")*t*t
        loss = (1-kd_weight)*ce/denominator + kd_weight*kd/len(rows)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError("Nonfinite training loss. A safe checkpoint will be retained.")
        scaler.scale(loss).backward()
        ce_sum += float(ce.detach())
        kd_sum += float(kd.detach())
        total_loss += float(loss.detach())
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                          config["gradient_clip"])
    if not scaler.is_enabled() and not bool(torch.isfinite(norm)):
        raise FloatingPointError("Nonfinite FP32 gradients.")
    old_scale = scaler.get_scale()
    if not bool(torch.isfinite(norm)):
        if scaler.is_enabled():
            scaler.update(new_scale=max(old_scale/2, 1.))
            optimizer.zero_grad(set_to_none=True)
            runtime["phase"] = "commit"
            return {"skipped": True}
        raise FloatingPointError("Nonfinite gradients.")
    runtime["phase"] = "step"
    scaler.step(optimizer)
    scaler.update()
    skipped = scaler.get_scale() < old_scale
    runtime["phase"] = "commit"
    optimizer.zero_grad(set_to_none=True)
    return {"loss": total_loss, "ce_sum": ce_sum, "ce_denominator": denominator,
            "kd_sum": kd_sum, "cases": len(rows), "gradient_norm": float(norm), "skipped": skipped}


def initial_state():
    return {"epoch": 0, "cursor": 0, "order": None, "updates": 0, "samples_seen": 0,
        "best_f1": -1., "best_score": -1., "best_loss": 1e30, "bad_epochs": 0, "epoch_improved": False,
        "epoch_ce_sum": 0., "epoch_ce_denominator": 0., "epoch_kd_sum": 0., "epoch_cases": 0,
        "history": [], "status": "ready"}


def train(model, ctx, config, run_dir, teacher=None, resume_path=None, progress_callback=None):
    config = copy.deepcopy(config)
    run_dir = Path(run_dir)
    device = next(model.parameters()).device
    optimizer = make_optimizer(model, config)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and config["amp"])
    if config.get("selection_metric", "macro_f1") not in ("macro_f1", "roc_auc"):
        raise ValueError("SELECTION_METRIC must be macro_f1 or roc_auc.")
    ctx["_ema"] = TrainableEMA(model, config.get("ema_decay", .98)) if config.get("use_ema", False) else None
    state = load_last(model, optimizer, scaler, config, ctx, resume_path) if resume_path else initial_state()
    if not resume_path:
        state["initial_trainable_sha256"] = trainable_hash(model)
        state["initial_seed"] = config["seed"]
        state["initial_model_settings"] = copy.deepcopy(model.settings)
    write_json(run_dir/"initialization.json", {k: state[k] for k in
        ("initial_trainable_sha256", "initial_seed", "initial_model_settings")})
    if (run_dir/"best.pt").exists():
        best = torch.load(run_dir/"best.pt", map_location="cpu", weights_only=False)
        state["best_f1"] = best["metrics"]["macro_f1"]
        state["best_loss"] = best["metrics"]["cross_entropy"]
        state["best_score"] = best["metrics"][config.get("selection_metric", "macro_f1")]
        if best["epoch"] == state["epoch"]+1:
            state["epoch_improved"] = True
        del best
    if config["run_startup_probe"]:
        config["train_batch_cases"], probe = startup_probe(model, ctx["train"], config)
        write_json(run_dir/"startup_probe.json", probe)
        print("Effective case batch:", config["train_batch_cases"],
              "| stress-only train epoch estimate:", round(probe["stress_epoch_minutes"], 1), "minutes", flush=True)
    if config.get("target_effective_cases"):
        config["accumulation_steps"] = math.ceil(config["target_effective_cases"]/config["train_batch_cases"])
    print("Cases per optimizer update:", config["train_batch_cases"]*config["accumulation_steps"],
          "| selection:", config.get("selection_metric", "macro_f1"),
          "| EMA:", config.get("use_ema", False), flush=True)
    write_json(run_dir/"effective_settings.json", config)
    model.use_amp = config["amp"]
    runner = runner_for(model)
    weights = ctx["class_weights"].to(device) if ctx["class_weights"] is not None else None
    # A valid initial checkpoint also gives us a rollback target for an interrupted optimizer step.
    save_last(model, optimizer, scaler, state, config, ctx, run_dir)
    started = last_save = last_backup = time.monotonic()
    invocation_start = state["updates"]
    runtime = {"phase": "idle", "group_rng": None}
    reason, retries = "completed", 0
    try:
        while state["epoch"] < config["epochs"]:
            if (config["patience"] is not None and state["bad_epochs"] >= config["patience"]
                    and state["epoch"] >= config["minimum_epochs"]):
                reason = "early_stopped"
                break
            if config["max_run_hours"] is not None and time.monotonic()-started >= config["max_run_hours"]*3600:
                reason = "paused_time_budget"
                break
            limit = config.get("max_updates_this_invocation")
            if limit is not None and state["updates"]-invocation_start >= limit:
                reason = "paused_update_budget"
                break
            if state["order"] is None:
                state["order"] = np.random.default_rng(config["seed"]+state["epoch"]).permutation(len(ctx["train"])).tolist()
                state["cursor"] = 0
                state["epoch_improved"] = False
            if state["cursor"] == len(state["order"]):
                validation_event(model, ctx, config, state, run_dir, "epoch_end")
                state["bad_epochs"] = 0 if state["epoch_improved"] else state["bad_epochs"]+1
                state["history"].append({"kind": "epoch", "epoch": state["epoch"]+1,
                    "update": state["updates"],
                    "train_ce": state["epoch_ce_sum"]/max(state["epoch_ce_denominator"], 1e-9),
                    "train_kd": state["epoch_kd_sum"]/max(state["epoch_cases"], 1),
                    "effective_train_cases": state["epoch_cases"]})
                state.update(epoch=state["epoch"]+1, cursor=0, order=None,
                    epoch_ce_sum=0., epoch_ce_denominator=0., epoch_kd_sum=0., epoch_cases=0)
                save_last(model, optimizer, scaler, state, config, ctx, run_dir)
                last_save = time.monotonic()
                if (state["epoch"] >= config["minimum_epochs"] and config["patience"] is not None and
                        state["bad_epochs"] >= config["patience"]):
                    reason = "early_stopped"
                    break
                continue
            model.train()
            end = min(state["cursor"]+config["train_batch_cases"]*config["accumulation_steps"], len(state["order"]))
            indices = state["order"][state["cursor"]:end]
            rows = [ctx["train"][i] for i in indices]
            apply_schedule(optimizer, config, state["samples_seen"], len(ctx["train"]))
            t0 = time.monotonic()
            outcome = train_group(model, runner, rows, weights, teacher, optimizer, scaler, config, runtime)
            if outcome["skipped"]:
                retries += 1
                set_rng(runtime["group_rng"])
                runtime["phase"] = "idle"
                state["history"].append({"kind": "amp_retry", "update": state["updates"],
                                         "new_scale": scaler.get_scale()})
                if retries > config["amp_retries"]:
                    raise FloatingPointError("Repeated nonfinite gradients; resume with a lower LR or AMP disabled.")
                continue
            retries = 0
            if ctx.get("_ema") is not None:
                ctx["_ema"].update(model)
            state.update(cursor=end, updates=state["updates"]+1, samples_seen=state["samples_seen"]+len(rows),
                epoch_ce_sum=state["epoch_ce_sum"]+outcome["ce_sum"],
                epoch_ce_denominator=state["epoch_ce_denominator"]+outcome["ce_denominator"],
                epoch_kd_sum=state["epoch_kd_sum"]+outcome["kd_sum"],
                epoch_cases=state["epoch_cases"]+len(rows))
            runtime["phase"] = "idle"
            if state["updates"] % config["log_every_updates"] == 0:
                row = {"kind": "train", "epoch": state["epoch"]+1, "update": state["updates"],
                    "cursor": state["cursor"], "loss": outcome["loss"], "gradient_norm": outcome["gradient_norm"],
                    "seconds": time.monotonic()-t0,
                    "lrs": {g["name"]: g["lr"] for g in optimizer.param_groups}}
                state["history"].append(row)
                print("Epoch", row["epoch"], "|", state["cursor"], "/", len(ctx["train"]),
                      "| update", row["update"], "| loss", round(row["loss"], 4),
                      "| LR", row["lrs"], flush=True)
            if config["eval_every_updates"] and state["updates"] % config["eval_every_updates"] == 0:
                validation_event(model, ctx, config, state, run_dir, "periodic")
            if time.monotonic()-last_save >= config["save_every_minutes"]*60:
                save_last(model, optimizer, scaler, state, config, ctx, run_dir)
                last_save = time.monotonic()
                if progress_callback:
                    backup_due = time.monotonic()-last_backup >= config["backup_every_minutes"]*60
                    progress_callback(backup_due)
                    if backup_due:
                        last_backup = time.monotonic()
    except (KeyboardInterrupt, torch.cuda.OutOfMemoryError, FloatingPointError) as exc:
        reason = "interrupted" if isinstance(exc, KeyboardInterrupt) else "paused_error"
        print("Pausing safely:", type(exc).__name__, safe_error(exc), flush=True)
        model.zero_grad(set_to_none=True)
        if runtime["phase"] in ("step", "commit"):
            # The exact completion point of an interrupted optimizer call is unknown.
            state = load_last(model, optimizer, scaler, config, ctx, run_dir/"last.pt")
            print("Restored the last saved optimizer boundary.", flush=True)
        elif runtime["group_rng"] is not None and runtime["phase"] == "compute":
            set_rng(runtime["group_rng"])
        runtime["phase"] = "idle"
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    # Always evaluate a trained current state before leaving a deliberate time/update pause.
    if state["updates"] > 0 and (reason not in ("paused_error", "interrupted") or not (run_dir/"best.pt").exists()):
        validation_event(model, ctx, config, state, run_dir, "final_or_pause")
    state["status"] = reason
    state["elapsed_this_invocation_hours"] = (time.monotonic()-started)/3600
    save_last(model, optimizer, scaler, state, config, ctx, run_dir)
    if progress_callback:
        progress_callback(True)
    del optimizer, scaler, runner
    ctx.pop("_ema", None)
    model.zero_grad(set_to_none=True)
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return state, config


def selected_model(model, run_dir):
    saved = torch.load(Path(run_dir)/"best.pt", map_location="cpu", weights_only=False)
    if saved.get("trained_updates", 0) < 1:
        raise ValueError("No trained selected checkpoint is available.")
    load_trainable(model, saved["model"])
    model.eval()
    return {k: v for k, v in saved.items() if k != "model"}


def export_selected(model, ctx, config, run_dir):
    run_dir = Path(run_dir)
    selected = selected_model(model, run_dir)
    dev_logits, dev_timing = predict(model, ctx["dev"], config)
    labels = [d["gold"] for d in ctx["dev"]]
    default = metrics(labels, dev_logits, .5)
    trials = [metrics(labels, dev_logits, float(t)) for t in config["threshold_grid"]]
    tuned = max(trials, key=lambda r: (r["macro_f1"], -abs(r["threshold"]-.5)))
    threshold = tuned["threshold"] if config["tune_threshold"] else .5
    final_dev = metrics(labels, dev_logits, threshold)
    test_logits, test_timing = predict(model, ctx["test"], config)
    test_predictions = (probabilities(test_logits) >= threshold).astype(np.int64)
    result = [{"id": d["id"], "pred": int(p)} for d, p in zip(ctx["test"], test_predictions)]
    if len(result) != len(ctx["test"]) or len({r["id"] for r in result}) != len(result):
        raise AssertionError("Missing/repeated test IDs.")
    write_json(run_dir/"prediction.json", result)
    write_json(run_dir/"dev_truth.json", [{"id": evaluation_id(d), "gold": int(d["gold"])} for d in ctx["dev"]])
    dev_pred = (probabilities(dev_logits) >= threshold).astype(np.int64)
    write_json(run_dir/"dev_prediction.json", [
        {"id": evaluation_id(d), "pred": int(p)} for d, p in zip(ctx["dev"], dev_pred)])
    for name, rows, logits in (("dev", ctx["dev"], dev_logits), ("test", ctx["test"], test_logits)):
        np.savez_compressed(run_dir/(name+"_probabilities.npz"), logits=logits,
            probability_1=probabilities(logits), row_uids=np.asarray([d["row_uid"] for d in rows]),
            original_ids=np.asarray([d["id"] for d in rows]))
    summary = {"selected_epoch": selected["epoch"], "selected_update": selected["trained_updates"],
        "selection_metric": selected.get("selection_metric", "macro_f1"),
        "evaluated_weights": selected.get("evaluated_weights", "raw"),
        "selected_checkpoint_metric": selected["metrics"], "dev_at_0_5": default,
        "dev_at_selected_threshold": final_dev, "selected_threshold": threshold,
        "threshold_trials": trials, "dev_timing": dev_timing, "test_timing": test_timing,
        "test_cases": len(result), "best_checkpoint_sha256": sha256(run_dir/"best.pt"),
        "training_unit": "one provided/selected case row; no per-chunk labels"}
    if ctx["train_eval"]:
        train_logits, timing = predict(model, ctx["train_eval"], config)
        summary["train_eval_subset"] = metrics([d["gold"] for d in ctx["train_eval"]], train_logits, threshold)
        summary["train_eval_subset_cases"] = len(ctx["train_eval"])
        summary["train_eval_timing"] = timing
    write_json(run_dir/"selected_results.json", summary)
    write_json(run_dir/"inference_settings.json", {"architecture": config["architecture"],
        "model_settings": model.settings, "document_selection": geometry(config), "threshold": threshold,
        "phase1_sha256": ctx["info"]["phase1_sha256"], "tokenizer_signature": ctx["signature"]})
    output = Path(config["output_root"])/(config["run_id"]+"_submission.zip")
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(run_dir/"prediction.json", "prediction.json")
    with zipfile.ZipFile(output) as z:
        assert z.namelist() == ["prediction.json"]
    summary["submission_zip"] = str(output)
    return summary


def export_teacher_cache(model, ctx, config, run_dir):
    run_dir = Path(run_dir)
    selected = selected_model(model, run_dir)
    rows = [d for d in ctx["rows"] if d["split"] == "train"]
    logits, timing = predict(model, rows, config)
    meta = {"purpose": "trained_cjpe_teacher", "split": "train", "class_order": [0, 1],
        "architecture": config["architecture"], "run_id": config["run_id"],
        "trained_updates": selected["trained_updates"], "selected_dev_macro_f1": selected["metrics"]["macro_f1"],
        "teacher_checkpoint_sha256": sha256(run_dir/"best.pt"),
        "phase1_sha256": ctx["info"]["phase1_sha256"],
        "cache_fingerprint": ctx["info"]["data_identity"]["cache_fingerprint"],
        "temperature_independent_logits": True, "document_selection": geometry(config), "timing": timing}
    path = run_dir/"teacher_logits_train.npz"
    temporary = path.with_name(path.name+".tmp")
    with open(temporary, "wb") as f:
        np.savez_compressed(f, row_uids=np.asarray([d["row_uid"] for d in rows]),
            logits=logits, labels=np.asarray([d["gold"] for d in rows], dtype=np.int64),
            metadata=np.asarray(json.dumps(meta)))
    temporary.replace(path)
    output = Path(config["output_root"])/(config["run_id"]+"_teacher_cache.zip")
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, path.name)
    return str(output)


In [ ]:
%%writefile /kaggle/working/cs781_phase4_train_src/cjpe_train_run.py
"""Kaggle orchestration, resumable run archives and report-ready exports."""
import copy
import gc
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
import torch
from cjpe_data import read_json, write_json, sha256, safe_error
from cjpe_train_io import (context, build_classifier, resume_files, extract_own_zip,
                           read_teacher_cache, load_trainable)
from cjpe_trainer import train, export_selected, export_teacher_cache


def plots(run_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    history_path = Path(run_dir)/"history.json"
    if not history_path.exists():
        return
    history = read_json(history_path)
    evaluations = [r for r in history if r.get("kind") == "validation"]
    training = [r for r in history if r.get("kind") == "train"]
    if not evaluations and not training:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    if training:
        axes[0].plot([r["update"] for r in training], [r["loss"] for r in training], label="training objective")
    if evaluations:
        axes[0].plot([r["update"] for r in evaluations],
                     [r["dev"]["cross_entropy"] for r in evaluations], label="dev CE")
        axes[1].plot([r["update"] for r in evaluations],
                     [r["dev"]["macro_f1"] for r in evaluations], label="dev macro-F1 at 0.5")
        ranked = [r for r in evaluations if r["dev"].get("roc_auc") is not None]
        if ranked:
            axes[2].plot([r["update"] for r in ranked],
                         [r["dev"]["roc_auc"] for r in ranked], label="dev ROC-AUC (selection)")
            axes[2].plot([r["update"] for r in ranked],
                         [r["dev"]["predicted_positive_fraction"] for r in ranked],
                         linestyle="--", label="predicted positive fraction at 0.5")
        training_evaluations = [r for r in evaluations if "train_eval_subset" in r]
        if training_evaluations:
            axes[1].plot([r["update"] for r in training_evaluations],
                [r["train_eval_subset"]["macro_f1"] for r in training_evaluations],
                marker="o", label="training evaluation macro-F1 at 0.5")
    axes[0].set(xlabel="Optimizer updates", ylabel="Loss")
    axes[1].set(xlabel="Optimizer updates", ylabel="Macro-F1", ylim=(0, 1))
    axes[2].set(xlabel="Optimizer updates", ylabel="AUC / positive fraction", ylim=(0, 1))
    for axis in axes:
        if axis.lines:
            axis.legend()
    fig.tight_layout()
    fig.savefig(Path(run_dir)/"training_curves.png", dpi=140)
    plt.close(fig)


def artifacts(config, run_dir, backup=False):
    run_dir = Path(run_dir)
    output = Path(config["output_root"])
    output.mkdir(parents=True, exist_ok=True)
    diagnostics = output/(config["run_id"]+"_diagnostics.zip")
    temporary = diagnostics.with_name(diagnostics.name+".tmp")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(run_dir.rglob("*")):
            if p.is_file() and p.suffix in (".py", ".json", ".csv", ".png", ".md", ".txt", ".npz"):
                if p.name == "teacher_logits_train.npz":
                    continue
                z.write(p, config["run_id"]+"/"+str(p.relative_to(run_dir)))
    temporary.replace(diagnostics)
    result = {"diagnostics_zip": str(diagnostics)}
    if backup:
        target = output/(config["run_id"]+"_backup.zip")
        temporary = target.with_name(target.name+".tmp")
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_STORED) as z:
            for p in sorted(run_dir.rglob("*")):
                if (p.is_file() and "__pycache__" not in p.parts and not p.name.endswith(".tmp")
                        and p.name != "teacher_logits_train.npz"):
                    z.write(p, config["run_id"]+"/"+str(p.relative_to(run_dir)))
        temporary.replace(target)
        result["backup_zip"] = str(target)
    return result


def run_training(config, source_dir):
    print("Starting CJPE setup; loading the shared input data.", flush=True)
    config = copy.deepcopy(config)
    source_dir = Path(source_dir)
    if config["action"] == "prepare_only":
        from cjpe_train_io import prepared_root
        from cjpe_reuse import load_prepared_cases
        rows, identity = load_prepared_cases(prepared_root(config),
            Path(config["work_root"])/"prepared_fast", Path(config["output_root"])/"phase4_cjpe_fast_cache.zip",
            config.get("data_read_workers", 16))
        if identity["cache_fingerprint"] != read_json(source_dir/"cjpe_reference.json")["prepared_cache_fingerprint"]:
            raise ValueError("Choose the matching successful CJPE preparation.")
        return {"run_id": config["run_id"], "status": "prepared_only", "cases": len(rows)}, {
            "fast_cache_zip": str(Path(config["output_root"])/"phase4_cjpe_fast_cache.zip")}
    run_dir = Path(config["work_root"])/"runs"/config["run_id"]
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir/"sources").mkdir(exist_ok=True)
    for path in source_dir.glob("*"):
        if path.suffix in (".py", ".json"):
            shutil.copy2(path, run_dir/"sources"/path.name)
    summary = {"run_id": config["run_id"], "architecture": config["architecture"],
               "status": "starting", "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    model = None
    paths = {}
    try:
        write_json(run_dir/"requested_settings.json", config)
        device = torch.device(config.get("device", "cuda:0"))
        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("Select Kaggle T4 x2.")
        if config["require_two_gpus"] and torch.cuda.device_count() < 2:
            raise RuntimeError("Select T4 x2, or explicitly disable REQUIRE_TWO_GPUS.")
        if min(config["epochs"], config["train_batch_cases"], config["eval_batch_cases"],
               config["accumulation_steps"]) < 1:
            raise ValueError("Epochs, batch sizes and accumulation must be positive.")
        torch.set_num_threads(2)
        print("GPU availability check passed; reading inputs.", flush=True)
        ctx = context(config, source_dir)
        write_json(run_dir/"input_record.json", ctx["info"])
        write_json(run_dir/"environment.json", {"torch": torch.__version__, "cuda": torch.version.cuda,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
        print("Run:", config["run_id"], "| architecture:", config["architecture"], flush=True)
        print("Training rows:", ctx["info"]["all_train_rows"], "provided ->",
              ctx["info"]["effective_train_rows"], "selected | dev:", len(ctx["dev"]),
              "rows /", ctx["info"]["unique_dev_forward_passes"], "unique forward inputs", flush=True)
        resume = resume_files(config, run_dir)
        settings = None
        if config["action"] == "export_only":
            saved = torch.load(run_dir/"best.pt", map_location="cpu", weights_only=False)
            if (saved["architecture"] != config["architecture"] or
                    saved["phase1_sha256"] != ctx["info"]["phase1_sha256"] or
                    saved["data_identity"]["cache_fingerprint"] != ctx["info"]["data_identity"]["cache_fingerprint"]):
                raise ValueError("Selected checkpoint architecture or data identity differs.")
            settings = saved["model_settings"]
            # Selected-checkpoint inference uses its actual training input geometry.
            for key in ("total_tokens", "overlap", "max_chunks", "policy"):
                config[key] = saved["config"][key]
            del saved
        print("Constructing the case classifier from original Phase 1 weights...", flush=True)
        model, model_record = build_classifier(ctx, config, settings)
        print("Moving classifier to", str(device), flush=True)
        model.to(device)
        model.use_amp = config["amp"]
        write_json(run_dir/"model_record.json", model_record)
        print("Parameters:", model_record["parameters"]["total_parameters"],
              "| trainable:", model_record["parameters"]["trainable_parameters"], flush=True)
        teacher = None
        if config["architecture"] == "efficient" and config["student_mode"] == "kd" and config["action"] == "train":
            supplied = config["teacher_cache_input"]
            if not supplied:
                raise ValueError("Set TEACHER_CACHE_INPUT to a genuine trained-teacher cache for KD.")
            supplied = Path(supplied)
            if supplied.suffix.lower() == ".zip":
                supplied = extract_own_zip(supplied, Path(config["work_root"])/"teacher_cache")
            teacher, meta = read_teacher_cache(supplied, ctx["rows"], ctx["info"]["data_identity"])
            write_json(run_dir/"teacher_identity.json", meta)
        if config["action"] == "train":
            state, config = train(model, ctx, config, run_dir, teacher, resume,
                lambda backup: artifacts(config, run_dir, backup=backup))
            summary["training_status"] = state["status"]
            summary["updates"] = state["updates"]
            summary["completed_epochs"] = state["epoch"]
        elif config["action"] != "export_only":
            raise ValueError("ACTION must be 'train' or 'export_only'.")
        if not (run_dir/"best.pt").exists():
            raise RuntimeError("No selected checkpoint yet. Resume training; the last optimizer checkpoint is retained.")
        # Numeric settings may change during a continuation. Export must reconstruct the
        # selected checkpoint's actual adapters, dropout/scale and document representation.
        chosen = torch.load(run_dir/"best.pt", map_location="cpu", weights_only=False)
        inference_config = copy.deepcopy(config)
        for key in ("total_tokens", "overlap", "max_chunks", "policy", "amp"):
            inference_config[key] = chosen["config"][key]
        selected_settings = chosen["model_settings"]
        del chosen, model
        model = None
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        model, _ = build_classifier(ctx, inference_config, selected_settings)
        model.to(device)
        model.use_amp = inference_config["amp"]
        summary["selected"] = export_selected(model, ctx, inference_config, run_dir)
        # Check exactly the format that will be submitted with the official scorer.
        scored = subprocess.run([sys.executable, str(source_dir/"evaluate_phase4.py"),
            "--truth", str(run_dir/"dev_truth.json"), "--prediction", str(run_dir/"dev_prediction.json")],
            capture_output=True, text=True, check=True, timeout=120)
        summary["official_evaluator_output"] = scored.stdout
        reported = next(float(s.split(":")[1]) for s in scored.stdout.splitlines() if s.startswith("Macro-F1:"))
        if abs(reported-summary["selected"]["dev_at_selected_threshold"]["macro_f1"]) > 1e-6:
            raise AssertionError("Official and local evaluation disagree.")
        if config["export_teacher_cache"]:
            summary["teacher_cache_zip"] = export_teacher_cache(model, ctx, inference_config, run_dir)
        summary["status"] = "exported"
        plots(run_dir)
        (run_dir/"README.md").write_text(
            "# CJPE run\n\n"
            "best.pt contains selected trainable parameters; last.pt contains resumable optimizer/RNG/progress state.\n"
            "Both require the matching original Phase 1 checkpoint. Keep the original prep tokenizer for future raw-text use.\n"
            "Training duplicates can be selected once; dev predictions expand to all provided row_uids before scoring.\n"
            "Test IDs are unchanged. Check input_record.json, effective_settings.json and selected_results.json.\n"
            "See selected_results.json for checkpoint selection metric and raw/EMA weight provenance; final threshold tuning is separate.\n"
            "Ordinary numeric hyperparameters may change on resume; model shape/data selection changes need a new run.\n")
    except KeyboardInterrupt:
        summary.update(status="interrupted", error="Interrupted during setup/evaluation/export. Saved checkpoints remain available.")
    except Exception as exc:
        summary.update(status="failed", error_type=type(exc).__name__, error=safe_error(exc))
        print("RUN ERROR:", safe_error(exc), flush=True)
    finally:
        summary["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        write_json(run_dir/"run_summary.json", summary)
        paths = artifacts(config, run_dir, backup=(run_dir/"last.pt").exists() or (run_dir/"best.pt").exists())
        if "selected" in summary:
            paths["submission_zip"] = summary["selected"]["submission_zip"]
        if "teacher_cache_zip" in summary:
            paths["teacher_cache_zip"] = summary["teacher_cache_zip"]
        packed_zip = Path(config["output_root"])/"phase4_cjpe_fast_cache.zip"
        if packed_zip.exists():
            paths["fast_cache_zip"] = str(packed_zip)
        print("STATUS:", summary["status"], flush=True)
        for key, value in paths.items():
            print(key, ":", value, flush=True)
        if model is not None:
            model.zero_grad(set_to_none=True)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return summary, paths


In [ ]:
(SOURCE_DIR / "cjpe_reference.json").write_text('{\n  "tokenizer": {\n    "path": "/kaggle/input/datasets/ash9201k/prep-2/cs781_phase1/preparation/prep_002_mixed16k/tokenizer_state.pkl",\n    "sha256": "080c6d2ce49a5c9ccfdd707145ec2b49c8a3f3d2e676e30d9c6e93c948900d9c",\n    "signature": {\n      "vocab_size": 16000,\n      "training_text_sha256": "f947121e9e0ddcfe2143a87ca331853bd6b5b5c01d65b7a35ca900eeba60649a",\n      "algorithm_source_sha256": "2131d6b5b41e2dbe1246ff6963daa63bef5d37b1de950b9adac6f103668619bb",\n      "preparation_configuration_sha256": "df137d6a56338b5e77a217ab6d525a7fe209804d405245c02ddd0f30e0ca6b18"\n    },\n    "vocab_size": 16000,\n    "reserved_ids": {\n      "pad": 0,\n      "unknown": 1,\n      "bos": 2,\n      "eos": 3\n    },\n    "round_trip_checks": "passed",\n    "retrained": false\n  },\n  "official_data_identity": {\n    "repo_id": "Exploration-Lab/CS-781-Capstone",\n    "revision": "93b0059be756c6c461f3a225de25aabda7d5f84a",\n    "resolved_files": {\n      "train": "Phase_4_CJPE/Train/train.json",\n      "dev": "Phase_4_CJPE/Train/dev.json",\n      "test": "Phase_4_CJPE/Test/test.json"\n    },\n    "provenance": "official pinned download",\n    "sha256": {\n      "Train/train.json": "a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72",\n      "Train/dev.json": "aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514",\n      "Test/test.json": "f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59"\n    }\n  },\n  "phase1_inputs": {\n    "decoder": {\n      "dataset": "decoder-004",\n      "hint": "decoder_004_mixed217m",\n      "sha256": "fd5add7ccc1d519a6f547fbd9a5c86a53276db8e09fb93c165d96531685d0bb7",\n      "source": "phase1_decoder.py",\n      "source_sha256": "c40066ffae9e41ee87d884450e0be64170eb5be49b682e3041796ddca6574a96",\n      "method": "upper_layers_and_final_norm",\n      "status": "passed",\n      "path": "/kaggle/input/datasets/ash9201k/decoder-004/cs781_phase1/runs/decoder_004_mixed217m/best_model.pt",\n      "bytes": 871513977,\n      "native_source_present": true,\n      "embedded_original_source_verified": true\n    },\n    "encoder_decoder": {\n      "dataset": "encdec-2",\n      "hint": "encdec_002_mixed217m_stable",\n      "sha256": "2a2d34a8df48bce7b6ab8132ea011803e2d044df8eaacde110046653b85516a5",\n      "source": "architecture_models.py",\n      "source_sha256": "24f762e290b90a47ed54bd7314950ef2883d77b56d6f4fce0a02f39bf24a1e04",\n      "method": "bottleneck_adapters_and_layer_norm",\n      "status": "passed",\n      "path": "/kaggle/input/datasets/ash9201k/encdec-002/cs781_phase1/runs/encdec_002_mixed217m_stable/best_model.pt",\n      "bytes": 871503069,\n      "native_source_present": true,\n      "embedded_original_source_verified": true\n    },\n    "prefix": {\n      "dataset": "prefix-001",\n      "hint": "prefix_001_mixed217m",\n      "sha256": "5308feee0a0df36b6609979002e5709de1ad1ecf647923be2e07752b1481ddd0",\n      "source": "architecture_models.py",\n      "source_sha256": "24f762e290b90a47ed54bd7314950ef2883d77b56d6f4fce0a02f39bf24a1e04",\n      "method": "output_side_lorma_plus",\n      "status": "passed",\n      "path": "/kaggle/input/datasets/ash9201k/prefix-001/cs781_phase1/runs/prefix_001_mixed217m/best_model.pt",\n      "bytes": 871521847,\n      "native_source_present": true,\n      "embedded_original_source_verified": true\n    },\n    "efficient": {\n      "dataset": "efficient-002",\n      "hint": "efficient",\n      "sha256": "7117b452f4ba76bf8590a633c6b2bfe7472a328aefed7127de171d73b68bf3ea",\n      "source": "architecture_models.py",\n      "source_sha256": "4f2db692383361b59ca1e606ef8cddebb9d1d64fa7ace806b7d5766a735c9937",\n      "method": "compact_student_gold_and_distillation",\n      "status": "passed",\n      "path": "/kaggle/input/datasets/ash9201k/efficient-002/cs781_phase1/runs/efficient_002_mixed217m_500m/best_model.pt",\n      "bytes": 871562425,\n      "native_source_present": true,\n      "embedded_original_source_verified": true\n    }\n  },\n  "prepared_cache_fingerprint": "813f30932f28a953687c38589a3b16fdad715506cb1dd6b9398b4667875c2075",\n  "prepared_status": "passed",\n  "counts": {\n    "train": 37387,\n    "dev": 1988,\n    "test": 1517\n  }\n}', encoding="utf-8")
(SOURCE_DIR / "local_training_validation.json").write_text('{\n  "status": "passed",\n  "revision": "1.2.0",\n  "torch": "2.5.1+cpu",\n  "fixture_models_only": true,\n  "full_dataset_training_tested": false,\n  "gpu_tested_locally": false,\n  "checks": {\n    "architecture_math_helpers": "passed",\n    "ema_values_and_exception_restoration": "passed",\n    "decoder_train_ema_select_export_plot": "passed",\n    "encoder_decoder_train_ema_select_export_plot": "passed",\n    "prefix_train_ema_select_export_plot": "passed",\n    "efficient_train_ema_select_export_plot": "passed",\n    "exact_resume_raw_and_ema_with_dropout": "passed",\n    "genuine_teacher_cache_and_matched_gold_kd_initialization": "passed",\n    "auc_selection_independent_of_fixed_threshold_f1": "passed",\n    "effective_batch_preserved_after_startup_reduction": "passed",\n    "incompatible_resume_protocol_rejected": "passed"\n  }\n}\n', encoding="utf-8")


## 3. Train and export

I trained on the official CJPE train split with identical case ID, text and label
duplicates counted once, leaving 32,305 training rows. Evaluation retains all
1,988 dev rows. Identical dev inputs share one forward pass, and the scorer
receives every supplied row.

The classifier reads up to four 512-token windows from the end of each case
with 64-token content overlap. It concatenates mean content states with a
backbone-specific readout and normalized chunk positions. A projection,
bidirectional GRU and attention pooling produce one vector for the binary classifier.

Training uses AdamW, a 10% warmup followed by cosine decay, and a target of
64 cases per optimizer update. Mixed precision and gradient checkpointing
bound GPU memory use. Dev ROC-AUC selects the checkpoint using an exponential
moving average of trainable parameters with decay 0.98.

Export reloads the selected weights and chooses one threshold on dev from 0.20
through 0.80. That threshold is fixed for all 1,517 test predictions. The
submission ZIP contains one root-level `prediction.json`.

### Replay the archived schedule

Run all cells with `STAGE = 1`, then set `STAGE = 2` and rerun from the settings
cell. Repeat with `STAGE = 3`. Keep the working directory and run ID between
stages. The update limits replay the archived pause points across different
hardware timings.

| Stage | Starting update | End of stage | Epoch cap | Patience |
| --- | ---: | --- | ---: | ---: |
| 1 | 0 | Update 1,011 | 4 | 1 |
| 2 | 1,011 | Update 2,020 | 4 | 1 |
| 3 | 2,020 | Early stopping or budget | 10 | 2 |

If a stage is interrupted, keep its settings and set
`MAX_UPDATES_THIS_INVOCATION` to the remaining updates before its boundary.
For a new Kaggle session, mount the matching KD backup and set `RESUME_INPUT`.
To export a saved KD checkpoint, use `ACTION = "export_only"` and the backup.

In [ ]:
# Rerunning after edits reloads the source cells and resumes this RUN_ID by default.
print("Importing the shared training modules...", flush=True)
for name in list(sys.modules):
    if name.startswith(("cjpe_", "phase1_")) or name == "a1_tokenizer":
        del sys.modules[name]
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
from cjpe_train_run import run_training
RUN_SUMMARY, OUTPUT_PATHS = run_training(CONFIG, SOURCE_DIR)
print(json.dumps({k: RUN_SUMMARY[k] for k in ("status", "training_status", "updates", "completed_epochs", "error")
                  if k in RUN_SUMMARY}, indent=2))

## 4. Selected-checkpoint results

The archived run paused at updates 1,011 and 2,020. I increased the epoch cap from 4 to 10 and patience from 1 to 2 at the second boundary. Training early-stopped after seven completed epochs and 3,535 updates. The last invocation recorded 2.02 hours inside the training loop.

The metrics cell reports dev F1 at 0.5 and at the selected threshold, the confusion matrix and scores on a fixed 512-case training subset. The curves use the 0.5 threshold for F1 and show AUC separately.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown

run_dir = WORK_ROOT/"runs"/RUN_ID
if ACTION == "prepare_only":
    print("Data-only preparation complete; download phase4_cjpe_fast_cache.zip in the final cell.")
summary_path = run_dir/"run_summary.json"
if summary_path.exists() and ACTION != "prepare_only":
    current_summary = json.loads(summary_path.read_text())
    print("Export status:", current_summary["status"], "| Training:", current_summary.get("training_status", ACTION))
    if current_summary.get("error"):
        print("Error:", current_summary["error"])
results_path = run_dir/"selected_results.json"
if results_path.exists() and ACTION != "prepare_only":
    results = json.loads(results_path.read_text())
    print("Selected epoch:", results["selected_epoch"], "| update:", results["selected_update"],
          "| threshold:", results["selected_threshold"])
    score_rows = []
    for label, key in (("Dev at 0.5", "dev_at_0_5"), ("Dev at selected threshold", "dev_at_selected_threshold"),
                       ("Training evaluation", "train_eval_subset")):
        if key in results:
            score_rows.append({"split": label, **{k: results[key].get(k) for k in
                ("macro_f1", "accuracy", "precision_1", "recall_1", "cross_entropy", "roc_auc", "predicted_positive_fraction", "threshold")}})
    display(pd.DataFrame(score_rows))
    display(pd.DataFrame(results["dev_at_selected_threshold"]["confusion_matrix"],
                         index=["true 0", "true 1"], columns=["pred 0", "pred 1"]))
    print("Test predictions:", results["test_cases"])
if (run_dir/"training_curves.png").exists() and ACTION != "prepare_only":
    display(Image(filename=str(run_dir/"training_curves.png")))

## 5. Reproduce or resume

The staged run settings above reproduce the recorded continuation protocol.

Use the same run ID with the saved `last.pt` to resume training. Set `RESUME_INPUT` to the matching backup when moving sessions. For an existing selected checkpoint, set `ACTION = "export_only"` and point `RESUME_INPUT` at its backup or `best.pt`. Use a fresh working directory for an independent run.

## 6. Artifacts

The submission ZIP is ready for CodaBench. Diagnostics retain the dev
and test probability exports used for analysis. The backup preserves
training state for later sessions and requires the matching original Phase 1
checkpoint when restored.

In [ ]:
from cjpe_train_run import artifacts
from IPython.display import FileLink

run_dir = WORK_ROOT/"runs"/RUN_ID
if run_dir.exists() and ACTION != "prepare_only":
    FINAL_PATHS = artifacts(CONFIG, run_dir,
        backup=(run_dir/"last.pt").exists() or (run_dir/"best.pt").exists())
    for suffix, key in (("_submission.zip", "submission_zip"), ("_teacher_cache.zip", "teacher_cache_zip")):
        candidate = OUTPUT_ROOT/(RUN_ID+suffix)
        if candidate.exists():
            FINAL_PATHS[key] = str(candidate)
    for key in ("submission_zip", "diagnostics_zip", "backup_zip", "teacher_cache_zip"):
        if key in FINAL_PATHS:
            path = Path(FINAL_PATHS[key])
            print(f"{key}: {path.name} ({path.stat().st_size/2**20:.2f} MiB)")
            display(FileLink(str(path)))
    print("Diagnostics record the run. The backup preserves the training state.")
else:
    FINAL_PATHS = {}
    print("No training run folder; prepare_only creates the shared fast-cache ZIP without model training.")
fast_zip = OUTPUT_ROOT/"phase4_cjpe_fast_cache.zip"
if fast_zip.exists():
    FINAL_PATHS["fast_cache_zip"] = str(fast_zip)
    print(f"Reusable data: {fast_zip.name} ({fast_zip.stat().st_size/2**20:.2f} MiB)")
    display(FileLink(str(fast_zip)))

## Method references

- Long-document judgment prediction. [ILDC for CJPE](https://aclanthology.org/2021.acl-long.313/)
- Temperature-scaled soft targets. [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)